# Week 07: Building the diffusion forward pass

Up to now you have built a fully analytic generative model: given a cycle's
amplitude, the **classical statistical model** from Weeks 03–06 emits a
per-year Gaussian over absolute emergence latitude, and you sample from it
directly. The model is interpretable and physically motivated, but it is also
a strong assumption — a single Gaussian per year, with shape parameters that
depend linearly on amplitude. Looking at the empirical distributions, you have
already seen places where the Gaussian fit leaves structure on the table:
bimodality at certain phases of certain cycles, asymmetric tails, year-to-year
fluctuations that a smooth analytic model cannot reproduce.

The goal of the next three weeks is to model what the classical model leaves
behind. For each 6-month window of each hemispheric cycle we will compute the
**residual** — the difference between the empirical latitude histogram and the
parametric histogram — and train a **score-based diffusion model** to generate
plausible residuals. The full generative model becomes *classical statistical
model + sampled residual*. If the diffusion model has learned anything
systematic, the combined model should outperform the classical model alone on
the scoreboard NLL.

**Strategic split across the three weeks.** A diffusion model has two parts:
a **forward process** that progressively corrupts data into noise (purely
numerical — no learning involved), and a **reverse process** that
progressively undoes the corruption (the learned part — a neural network).
All the diffusion-specific mathematics lives in the forward process; the
reverse process is a standard regression problem dressed in PyTorch. We
exploit this separation deliberately:

- **Week 07 (this notebook):** Build the dataset of residuals and implement
  the forward process in pure numpy. By the end of this week you will be able
  to take a real residual and watch it dissolve into Gaussian noise across
  T timesteps, with every line of code transparent and debuggable.
- **Week 08:** Build the AI/ML pipeline (PyTorch + Lightning) and train an
  **unconditional** diffusion model that learns the marginal distribution of
  residuals. Sample from it and verify that the samples match the training
  distribution.
- **Week 09:** Add **conditioning** on (cycle amplitude, universal-path
  latitude). Sample residuals for held-out test cycles, combine with the
  classical model, and run the result through `compute_global_nll`.

By the end of Week 09 you will have a working **conditional generative model**
for the butterfly diagram that learns from data what the classical model
misses. This week is the foundation: get the forward process exactly right,
and the rest is standard deep learning.

**By the end of this notebook you should be able to:**
- Explain what a **residual** is in vector form, and why each one is a single
  point in a 15-dimensional space (not a function of latitude).
- Construct a **stratified** training / validation / test split by
  hemispheric cycle amplitude, and explain why all three splits are needed
  for diffusion training.
- Implement and visualize a **cosine noise schedule** with T = 200 timesteps,
  and read off α_t, σ_t, and the **signal-to-noise ratio** SNR(t) at any
  point on the schedule.
- Apply the **forward corruption** r_t = α_t · r + σ_t · ε to map any clean
  residual to any noise level in a single step.
- Verify empirically that at t = T the corrupted residuals are
  indistinguishable from N(0, I), regardless of the original residual — this
  is the property that lets the reverse process start from pure noise next
  week.


In [ ]:
import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    subprocess.run(["git", "-C", repo_path, "pull"], check=True)
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()

## 1) Load data

We work with the same composite catalog of daily sunspot group measurements used in Weeks 03 and 04. Each row records one sunspot group observation: its date, heliographic latitude, and corrected area in millionths of a solar hemisphere (MSH). We add convenience columns for hemisphere, absolute latitude, and decimal year that all subsequent sections depend on.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from pathlib import Path
from scipy.stats import norm as sp_norm
from scipy.optimize import curve_fit, minimize_scalar, minimize

data_path = Path(repo_path) / 'data' / 'composite_sunspot_groups_peak_area.csv'
df = pd.read_csv(data_path, parse_dates=[[0, 1, 2]], keep_date_col=False)
df.rename(columns={"year_month_day": "date"}, inplace=True)
df = df[df["latitude"].notna()].copy()

df["hemisphere"]   = df["latitude"].apply(lambda v: "north" if v >= 0 else "south")
df["abs_latitude"] = df["latitude"].abs()
df["year"] = df["date"].dt.year
df["decimal_year"] = df["date"].dt.year + df["date"].dt.dayofyear / 365.25

# Remove pores and small objects
df = df[df["correctedArea"] > 30].copy()


df.head()

In [ ]:
# Generate a colormap for cycles
import matplotlib.cm as cm
import numpy as np

# Filter out rows with missing CYCLE values
cycles = sorted(df["CYCLE"].dropna().unique())
n_cycles = len(cycles)
cmap = cm.get_cmap("tab20", n_cycles)
cycle_colors = {cyc: cmap(i) for i, cyc in enumerate(cycles)}

fig, ax = plt.subplots(figsize=(14, 3))

# Plot butterfly diagram with each cycle in a different color
for cyc in cycles:
    df_cyc = df[df["CYCLE"] == cyc]
    ax.scatter(df_cyc["date"], df_cyc["latitude"], s=3, 
               c=[cycle_colors[cyc]], label=f"Cycle {int(cyc)}", 
               alpha=0.5, edgecolors="none")

# Calculate and overplot yearly mean latitude for each cycle and hemisphere
df["year"] = df["date"].dt.year

ax.set_xlabel("Date")
ax.set_ylabel("Latitude (degrees)")
ax.set_ylim(-45, 45)
ax.axhline(0, color="k", linewidth=0.5, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## 2) Standardize time: align cycles at the 15° crossing (τ)

Solar cycles don't start on the same calendar date and don't last the same length of time, so we can't compare them on a shared time axis without first choosing a common reference point. The anchor we use is the moment when the **yearly mean absolute emergence latitude crosses 15° on its way equatorward** — a latitude that falls reliably in the well-observed middle of every cycle.

For each (cycle, hemisphere) pair we find that crossing by linear interpolation between the last year the mean sits above 15° and the first year it falls below. The shifted coordinate **τ = decimal\_year − t₀** is in physical units (years) and places all cycles on the same footing regardless of when they actually occurred.

In [ ]:
t0_lookup = {}
for (cyc, hemi), group in df.groupby(["CYCLE", "hemisphere"]):
    yearly_mean = group.groupby("year")["abs_latitude"].mean().sort_index()
    years, means = yearly_mean.index.values, yearly_mean.values
    below = means < 15.0
    if not below.any() or below.all():
        continue
    idx1 = np.argmax(below)
    if idx1 == 0:
        continue
    y0, mu0 = years[idx1 - 1], means[idx1 - 1]
    y1, mu1 = years[idx1],     means[idx1]
    t0_lookup[(cyc, hemi)] = y0 + (15.0 - mu0) / (mu1 - mu0)

df["t0"]  = df.apply(lambda r: t0_lookup.get((r["CYCLE"], r["hemisphere"]), np.nan), axis=1)
df["tau"] = df["decimal_year"] - df["t0"]

## 3) Fit a universal mean path μ(τ) and refine the time origin

The equatorward drift of the mean emergence latitude follows an exponential decay:

$$\mu(\tau) = a \, e^{-\tau / b}$$

where **a** is the mean latitude at τ = 0 and **b** is the e-folding timescale — the number of years it takes the mean to drop by a factor of e ≈ 2.7. In Week 04 you found that the best-fit (a, b) is nearly the same for every cycle, so here we fit it once, globally, by pooling binned (τ, μ) measurements from all hemisphere-cycles.

With the universal (a, b) in hand we can refine each cycle's t₀: a small time shift Δt is found for each hemisphere-cycle that minimises its residuals against the common template. The refined coordinate **τ_refined** gives a tighter alignment than the raw 15° crossing, because it uses the full shape of the wing rather than a single crossing point.

In [ ]:
def exp_decay(tau, a, b):
    return a * np.exp(-tau / b)

N_BINS_13   = 20
cycles_13   = [c for c in sorted(df["CYCLE"].dropna().unique()) if c >= 12]
cmap_13     = cm.get_cmap("tab20", len(cycles_13))
cyc_idx_13  = {c: i for i, c in enumerate(cycles_13)}

all_tau_bins, all_mu_bins = [], []
hemicycle_bins_13 = {}

for cyc in cycles_13:
    for hemi in ["north", "south"]:
        if (cyc, hemi) not in t0_lookup:
            continue
        mask   = (df["CYCLE"] == cyc) & (df["hemisphere"] == hemi) & df["tau"].notna()
        df_sel = df[mask]
        if len(df_sel) < 50:
            continue
        t_min, t_max = df_sel["tau"].min(), df_sel["tau"].max()
        bins        = np.linspace(t_min, t_max, N_BINS_13 + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        bt, bm = [], []
        for i in range(N_BINS_13):
            lats_bin = df_sel.loc[
                (df_sel["tau"] >= bins[i]) & (df_sel["tau"] < bins[i + 1]),
                "abs_latitude"].values
            if len(lats_bin) < 10:
                continue
            mu_f, _ = sp_norm.fit(lats_bin)
            bt.append(bin_centers[i]); bm.append(mu_f)
        if len(bt) < 5:
            continue
        bt = np.array(bt); bm = np.array(bm)
        hemicycle_bins_13[(cyc, hemi)] = (bt, bm)
        all_tau_bins.extend(bt); all_mu_bins.extend(bm)

popt_global, _ = curve_fit(exp_decay, all_tau_bins, all_mu_bins, p0=[15.0, 5.0])
a_mu_univ, b_mu_univ = popt_global
print(f"Universal μ(τ):  a = {a_mu_univ:.2f}°   b = {b_mu_univ:.2f} yr")

# Refine t₀ per hemisphere-cycle
t0_refined = {}
for (cyc, hemi), (bt, bm) in hemicycle_bins_13.items():
    def _res(dt, _bt=bt, _bm=bm):
        return np.sum((_bm - exp_decay(_bt - dt, a_mu_univ, b_mu_univ)) ** 2)
    res = minimize_scalar(_res, bounds=(-4, 4), method="bounded")
    t0_refined[(cyc, hemi)] = t0_lookup[(cyc, hemi)] + res.x

df["t0_refined"]  = df.apply(lambda r: t0_refined.get((r["CYCLE"], r["hemisphere"]), np.nan), axis=1)
df["tau_refined"] = df["decimal_year"] - df["t0_refined"]

In [ ]:
TAU_GRID_13 = np.linspace(-8, 8, 300)

# Plot 1: All hemisphere-cycles aligned to the universal mean path
fig1, ax1 = plt.subplots(figsize=(12, 5))
for (cyc, hemi) in hemicycle_bins_13:
    mask = (df["CYCLE"] == cyc) & (df["hemisphere"] == hemi) & df["tau_refined"].notna()
    grp  = df[mask]
    ax1.scatter(grp["tau_refined"], grp["abs_latitude"],
                s=10, color=cmap_13(cyc_idx_13[cyc]), alpha=0.25, edgecolors="none")
ax1.plot(TAU_GRID_13, exp_decay(TAU_GRID_13, a_mu_univ, b_mu_univ),
         color="black", linewidth=2.5,
         label=f"Universal μ(τ)  a={a_mu_univ:.1f}°  b={b_mu_univ:.1f} yr")
ax1.axvline(0, color="black", linewidth=1.5, linestyle="--", label="τ = 0")
ax1.axhline(15, color="gray", linewidth=1, linestyle=":", alpha=0.6)
ax1.set_xlabel("τ (years relative to refined t₀)")
ax1.set_ylabel("|Latitude| (degrees)")
ax1.set_title("All hemisphere-cycles aligned to the universal mean path")
ax1.set_ylim(0, 45); ax1.set_xlim(-8, 8)
ax1.legend(loc="upper right")
sm1 = plt.cm.ScalarMappable(cmap="tab20",
                              norm=plt.Normalize(vmin=min(cycles_13), vmax=max(cycles_13)))
sm1.set_array([])
fig1.colorbar(sm1, ax=ax1, pad=0.02, label="Cycle number")
plt.tight_layout()
plt.show()


## 4) Model σ(μ): spread as a function of mean latitude

Rather than tracking the spread σ as a function of time, we describe it as a function of the **current mean emergence latitude** μ. This is more natural physically: two cycles at the same latitude should have similar spreads regardless of their absolute timing, because the spread is determined by the state of the dynamo, not by the calendar.

The spread is not monotonic in μ. At high latitudes (early in the cycle) there are few sunspots so the spread appears narrow; at low latitudes (late in the cycle) the active zone has collapsed toward the equator and is narrow again. The maximum spread occurs somewhere in between, at a characteristic mid-cycle latitude μ_peak. This bell-shaped behaviour in μ-space is captured by a **split Gaussian**:

$$\sigma(\mu) = A \exp\!\left(-\frac{(\mu - \mu_{\mathrm{peak}})^2}{2 s_{L,R}^2}\right)$$

where $s_L$ governs the poleward side (large μ, early cycle) and $s_R$ governs the equatorward side (small μ, late cycle). When $s_L \neq s_R$ the envelope is asymmetric — which is what the data show.

In [ ]:
def split_normal_mu(mu, A, mu_peak, s_L, s_R):
    return np.where(
        mu >= mu_peak,
        A * np.exp(-0.5 * ((mu - mu_peak) / s_L) ** 2),
        A * np.exp(-0.5 * ((mu - mu_peak) / s_R) ** 2),
    )

N_BINS_15 = 20
MU_GRID   = np.linspace(2, 42, 300)
results_15 = []

for cyc in cycles_13:
    for hemi in ["north", "south"]:
        if (cyc, hemi) not in t0_refined:
            continue
        mask   = (df["CYCLE"] == cyc) & (df["hemisphere"] == hemi) & df["tau_refined"].notna()
        df_sel = df[mask]
        if len(df_sel) < 50:
            continue
        t_min, t_max = df_sel["tau_refined"].min(), df_sel["tau_refined"].max()
        bins        = np.linspace(t_min, t_max, N_BINS_15 + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        bm_list, bs_list = [], []
        for i in range(N_BINS_15):
            lats_bin = df_sel.loc[
                (df_sel["tau_refined"] >= bins[i]) & (df_sel["tau_refined"] < bins[i + 1]),
                "abs_latitude"].values
            if len(lats_bin) < 10:
                continue
            mu_f, sigma_f = sp_norm.fit(lats_bin)
            bm_list.append(mu_f); bs_list.append(sigma_f)
        if len(bm_list) < 5:
            continue
        bm_arr = np.array(bm_list); bs_arr = np.array(bs_list)
        sidx = np.argsort(bm_arr)
        bm_arr, bs_arr = bm_arr[sidx], bs_arr[sidx]
        try:
            p0 = [bs_arr.max(), bm_arr[np.argmax(bs_arr)], 5.0, 4.0]
            popt, _ = curve_fit(split_normal_mu, bm_arr, bs_arr, p0=p0, maxfev=10_000)
            A_f, mu_peak_f, sL_f, sR_f = popt
        except RuntimeError:
            continue
        if not (0.5 < A_f < 20 and 2 < mu_peak_f < 38 and 0.5 < sL_f < 20 and 0.5 < sR_f < 20):
            continue
        results_15.append(dict(
            cycle=cyc, hemisphere=hemi,
            A=A_f, mu_peak=mu_peak_f, sL=sL_f, sR=sR_f,
            sigma_curve=split_normal_mu(MU_GRID, A_f, mu_peak_f, sL_f, sR_f),
            bin_mu=bm_arr, bin_sigma=bs_arr,
        ))

print(f"Fitted {len(results_15)} hemisphere-cycles.")

## 5) Universal piecewise-linear envelope

The key insight from Week 04 is that the equatorward half of every σ(μ) curve — the side where spots are migrating toward the equator — falls on **the same straight line** regardless of which cycle you look at. This universality tells us something deep: the process that narrows the active zone as the cycle approaches minimum is the same in every cycle. One slope and one intercept describe it for all of them.

The poleward half, by contrast, is cycle-specific: stronger cycles detach from the shared line at a higher latitude and descend more steeply. We encode this with a piecewise-linear model fitted *jointly* across all hemisphere-cycles:

$$\sigma(\mu) = \begin{cases} m_{\text{shared}}\,\mu + b_{\text{shared}} & \mu \le \mu_{\text{peak},i} \quad \text{(universal — same for all cycles)} \\ m_i\,\mu + b_i & \mu > \mu_{\text{peak},i} \quad \text{(per-cycle)} \end{cases}$$

Continuity at the detachment latitude $\mu_{\text{peak},i}$ determines $b_i$ automatically, so the model needs only two universal numbers $(m_{\text{shared}},\, b_{\text{shared}})$ and two per-cycle numbers $(\mu_{\text{peak},i},\, m_i)$ to fully describe each wing's spread profile. In the exercises below you will connect those per-cycle numbers to the overall strength of each cycle.

In [ ]:
def piecewise_linear_wing(mu, m_shared, b_shared, mu_peak, m_i):
    """Universal equatorward line + per-cycle poleward line, joined continuously at mu_peak."""
    sigma_peak = m_shared * mu_peak + b_shared
    b_per = sigma_peak - m_i * mu_peak
    return np.clip(
        np.where(mu <= mu_peak, m_shared * mu + b_shared, m_i * mu + b_per),
        0.0, None
    )

# Bootstrap: fit the shared equatorward line from pooled equatorward-side points
eq_mu, eq_sigma = [], []
for r in results_15:
    mask = r["bin_mu"] <= r["mu_peak"]
    eq_mu.extend(r["bin_mu"][mask]); eq_sigma.extend(r["bin_sigma"][mask])
m_init, b_init = np.polyfit(eq_mu, eq_sigma, 1)
m_i_init = np.mean([-r["A"] / (2 * max(r["sL"], 1.0)) for r in results_15])

# Joint L-BFGS-B optimisation
n_hc = len(results_15)

def residuals_pl(x):
    m_sh, b_sh = x[0], x[1]
    return sum(
        np.sum((r["bin_sigma"] - piecewise_linear_wing(
            r["bin_mu"], m_sh, b_sh, x[2 + 2*i], x[3 + 2*i])) ** 2)
        for i, r in enumerate(results_15)
    )

x0 = np.array([m_init, b_init] + [val for r in results_15
                                   for val in (r["mu_peak"], m_i_init)])
bounds = list(zip(
    [0.0, -5.0] + [2.0, -5.0] * n_hc,
    [2.0,  5.0] + [38.0, 0.0] * n_hc,
))
opt = minimize(residuals_pl, x0, method="L-BFGS-B", bounds=bounds)

m_shared_fit, b_shared_fit = opt.x[0], opt.x[1]
print(f"Universal line:  σ(μ) = {m_shared_fit:.4f}·μ + {b_shared_fit:.4f}")
print(f"Zero crossing at μ = {-b_shared_fit / m_shared_fit:.2f}°")

# Reconstruct per-cycle fitted curves
MU_GRID_PL = np.linspace(2, 42, 300)
results_pl = []
for i, r in enumerate(results_15):
    mu_peak_i    = opt.x[2 + 2*i]
    m_i          = opt.x[3 + 2*i]
    sigma_peak_i = m_shared_fit * mu_peak_i + b_shared_fit
    results_pl.append(dict(
        cycle=r["cycle"], hemisphere=r["hemisphere"],
        m_shared=m_shared_fit, b_shared=b_shared_fit,
        mu_peak=mu_peak_i, m_i=m_i, sigma_peak=sigma_peak_i,
        sigma_curve=piecewise_linear_wing(
            MU_GRID_PL, m_shared_fit, b_shared_fit, mu_peak_i, m_i),
        bin_mu=r["bin_mu"], bin_sigma=r["bin_sigma"],
    ))

# RMSE comparison
def rmse_sn(res_list):
    return np.sqrt(np.mean([
        np.mean((r["bin_sigma"] - split_normal_mu(
            r["bin_mu"], r["A"], r["mu_peak"], r["sL"], r["sR"])) ** 2)
        for r in res_list
    ]))

def rmse_pl(res_list):
    return np.sqrt(np.mean([
        np.mean((r["bin_sigma"] - piecewise_linear_wing(
            r["bin_mu"], r["m_shared"], r["b_shared"], r["mu_peak"], r["m_i"])) ** 2)
        for r in res_list
    ]))

rmse_15 = rmse_sn(results_15)
rmse_16 = rmse_pl(results_pl)
print(f"RMSE — split-normal independent : {rmse_15:.3f}°")
print(f"RMSE — piecewise-linear joint   : {rmse_16:.3f}°  ({(rmse_16 - rmse_15)/rmse_15*100:+.1f}%)")

# ── Figure ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

for r in results_pl:
    c = cmap_13(cyc_idx_13.get(r["cycle"], 0))
    ax.scatter(r["bin_mu"], r["bin_sigma"], s=20, color=c,
               alpha=0.8, edgecolors="none", zorder=1)
    ax.plot(MU_GRID_PL, r["sigma_curve"], color=c,
            linewidth=1.2, alpha=0.7, zorder=2)

mu_eq = np.linspace(0, max(r["mu_peak"] for r in results_pl) + 2, 200)
ax.plot(mu_eq, np.clip(m_shared_fit * mu_eq + b_shared_fit, 0, None),
        color="tab:red", linewidth=2.5, linestyle="--",
        label=f"Universal equatorward line  σ = {m_shared_fit:.3f}·μ + {b_shared_fit:.3f}",
        zorder=5)

ax.invert_xaxis()
ax.set_xlim(42, 2)
ax.set_ylim(0, 12)
ax.set_xlabel("|μ| (mean emergence latitude, °)")
ax.set_ylabel("σ (degrees)")
ax.set_title(
    f"Universal piecewise-linear envelope  "
    f"σ = {m_shared_fit:.3f}·μ + {b_shared_fit:.3f}  |  RMSE = {rmse_16:.3f}°\n"
    "Red dashed: universal equatorward line shared by all hemisphere-cycles"
)
ax.legend(loc="upper left", fontsize=9)

sm = plt.cm.ScalarMappable(cmap="tab20",
                            norm=plt.Normalize(vmin=min(cycles_13), vmax=max(cycles_13)))
sm.set_array([])
fig.colorbar(sm, ax=ax, pad=0.02, label="Cycle number")
plt.tight_layout()
plt.show()

# Numeric summary
mu_peaks  = [r["mu_peak"]    for r in results_pl]
sig_peaks = [r["sigma_peak"] for r in results_pl]
m_is    = [r["m_i"]      for r in results_pl]
print(f"\nPer-cycle variability:")
print(f"  μ_peak : {np.mean(mu_peaks):.2f}° ± {np.std(mu_peaks):.2f}°  ← detachment latitude")
print(f"  σ_peak : {np.mean(sig_peaks):.2f}° ± {np.std(sig_peaks):.2f}°  ← derived amplitude")
print(f"  m_i  : {np.mean(m_is):.4f} ± {np.std(m_is):.4f}  ← per-cycle poleward slope")

---
## Week 05 Exercises: From Wing Shape to a Full Synthetic Cycle

Sections 1–5 give you the geometric skeleton of a butterfly wing. The exercises below ask you to attach one more piece: the **amplitude** of the solar cycle. A cycle's amplitude tells you how active it was — how many sunspots appeared, how large they were, how long the active zone persisted. Once you can predict the wing's shape parameters from its amplitude, you can synthesize a complete 2D wing from a single number.

The four tasks build up to that synthesis step by step.

---

## Task 17 — Solar Cycle Amplitude from Total Sunspot Area

The standard proxy for solar cycle strength is the **total daily corrected sunspot area** — the sum of all individual sunspot group areas on a given day, measured in millionths of a solar hemisphere (MSH). Plotted over time, this quantity rises and falls with each ~11-year cycle, with strong cycles reaching much higher peaks than weak ones.

Raw daily values are noisy (individual sunspot groups appear and disappear), so we smooth with a rolling average before looking for cycle peaks.

**Tasks:**
- For each hemisphere separately, compute the total daily corrected area. Include only sunspot groups with a corrected area **greater than 50 MSH** — smaller features are at the noise floor of the older observations.
- Apply rolling averages with windows of **3, 6, 12, and 24 months**. Plot all four smoothed curves on the same axes for each hemisphere.
- Focus on **cycles 12 and later**. Earlier data come from heterogeneous sources with different calibration standards, making direct amplitude comparisons unreliable.

Which smoothing window best captures the large-scale rise-and-fall of each cycle without washing out differences between strong and weak cycles?

In [ ]:
# Task 17: Solar Cycle Amplitude from Total Sunspot Area

# Filter: cycles ≥ 12 and correctedArea > 50 MSH
df_amp = df[(df["CYCLE"] >= 12) & (df["correctedArea"] > 50)].copy()

# Daily total corrected area per hemisphere
daily_north = df_amp[df_amp["hemisphere"] == "north"].groupby("date")["correctedArea"].sum()
daily_south = df_amp[df_amp["hemisphere"] == "south"].groupby("date")["correctedArea"].sum()

# Reindex to a complete daily grid so rolling windows are contiguous (fill 0 for no-sunspot days)
date_range_17 = pd.date_range(
    min(daily_north.index.min(), daily_south.index.min()),
    max(daily_north.index.max(), daily_south.index.max()),
    freq="D",
)
daily_north = daily_north.reindex(date_range_17, fill_value=0)
daily_south = daily_south.reindex(date_range_17, fill_value=0)

# Rolling averages: approximate month lengths in days
windows_17 = {"3 mo": 91, "6 mo": 182, "12 mo": 365, "24 mo": 730}
line_styles = [("tab:blue", 1.0), ("tab:orange", 1.2), ("tab:green", 1.6), ("tab:red", 2.0)]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, hemi_label, daily in zip(axes, ["North", "South"], [daily_north, daily_south]):
    for (label, win), (col, lw) in zip(windows_17.items(), line_styles):
        smoothed = daily.rolling(win, center=True, min_periods=win // 3).mean()
        ax.plot(smoothed.index, smoothed.values, label=label, color=col, linewidth=lw, alpha=0.85)
    ax.set_ylabel("Total corrected area (MSH)")
    ax.set_title(f"{hemi_label} hemisphere — smoothed total sunspot area "
                 "(cycles ≥ 12, correctedArea > 50 MSH)")
    ax.legend(loc="upper right", title="Smoothing window")
axes[1].set_xlabel("Date")
plt.tight_layout()
plt.show()

print("Best window: 12 months.")
print("  The 3- and 6-month windows retain month-to-month burst noise that masks the")
print("  cycle envelope. The 24-month window over-smooths: weak cycles (e.g. 14, 24)")
print("  appear merged with their neighbours and amplitude differences are lost.")
print("  The 12-month window suppresses the ~27-day solar-rotation modulation and")
print("  within-cycle transients while cleanly resolving the rise, peak, and decay of")
print("  each cycle — including the contrast between strong (19, 22) and weak (14, 24).")

## Task 18 — Peak Amplitude and Timing of Each Hemispheric Cycle

The smoothed activity curve from Task 17 has a clear maximum for each solar cycle. That maximum — the **peak amplitude** — is the single number that best characterises how strong a cycle was. Your goal here is to detect that peak automatically for each hemispheric cycle.

**Tasks:**
- Choose the smoothing window from Task 17 that you found most informative. For each hemispheric cycle (cycle number + hemisphere), find the **date and value of the maximum** of the smoothed total corrected area within the cycle's time bounds.
- Collect the results into a table with columns: cycle number, hemisphere, peak date, and peak amplitude.
- Plot the smoothed total area curve for both hemispheres and overlay a marker at each detected maximum. Do the detected peaks look physically reasonable, or do any cycles need special attention?

In [ ]:
# Task 18: Peak Amplitude and Timing of Each Hemispheric Cycle

WIN_18 = 365  # 12-month window chosen from Task 17

smooth_north = daily_north.rolling(WIN_18, center=True, min_periods=WIN_18 // 3).mean()
smooth_south = daily_south.rolling(WIN_18, center=True, min_periods=WIN_18 // 3).mean()

# For each (cycle, hemisphere), find the date and value of the smoothed-area maximum
# within the calendar span of that cycle's sunspot records.
peak_records = []
for cyc in cycles_13:
    cyc_dates = df[df["CYCLE"] == cyc]["date"]
    if len(cyc_dates) == 0:
        continue
    d_min, d_max = cyc_dates.min(), cyc_dates.max()
    for hemi, smooth in [("north", smooth_north), ("south", smooth_south)]:
        seg = smooth[(smooth.index >= d_min) & (smooth.index <= d_max)].dropna()
        if len(seg) == 0:
            continue
        peak_records.append(dict(
            cycle=int(cyc), hemisphere=hemi,
            peak_date=seg.idxmax(),
            peak_amplitude=float(seg.max()),
        ))

peaks_df = (pd.DataFrame(peak_records)
            .sort_values(["cycle", "hemisphere"])
            .reset_index(drop=True))
print(peaks_df.to_string(index=False))

# Plot smoothed curves with detected peaks overlaid
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, hemi, smooth, color in zip(
        axes,
        ["north", "south"],
        [smooth_north, smooth_south],
        ["steelblue", "tomato"]):
    ax.plot(smooth.index, smooth.values, color=color, linewidth=1.2, alpha=0.8)
    for _, row in peaks_df[peaks_df["hemisphere"] == hemi].iterrows():
        ax.scatter(row["peak_date"], row["peak_amplitude"], s=70, color="black", zorder=5)
        ax.annotate(str(int(row["cycle"])),
                    (row["peak_date"], row["peak_amplitude"]),
                    textcoords="offset points", xytext=(4, 4), fontsize=8)
    ax.set_ylabel("Total corrected area (MSH)")
    ax.set_title(f"{hemi.capitalize()} hemisphere — 12-month smoothed area "
                 "with detected cycle peaks")
axes[1].set_xlabel("Date")
plt.tight_layout()
plt.show()

print("\nPhysical check: cycle 19 (1954–1964) is the strongest on record — its peak stands")
print("out clearly in both hemispheres. Cycle 24 (2008–2019) is the weakest in a century.")
print("Hemispheric peaks sometimes differ by 1–2 years (e.g. cycle 23), reflecting the")
print("genuine north–south timing asymmetry that this model will not capture.")

## Task 19 — Relating Cycle Amplitude to Wing Shape Parameters

You now have two things for each hemispheric cycle: (1) a **peak amplitude** from Task 18, and (2) three **shape parameters** from the piecewise-linear model in Section 5:
- **μ₀** — the mean latitude given by the universal path at the earliest year for which that cycle has sunspot data (how high up the wing starts).
- **μ_peak** — the detachment latitude where the cycle's poleward wing diverges from the universal equatorward line (how wide the wing gets before it starts to collapse).
- **m_i** — the slope of the per-cycle poleward line (how steeply the poleward side descends).

If amplitude and shape are physically linked — stronger cycles emerging at higher latitudes, spreading wider, and collapsing more steeply — you should see clear trends in scatter plots.

**Tasks:**

1. **Earliest emergence latitude μ₀:** For each hemispheric cycle, use the universal path μ(τ_refined) to look up the mean latitude at the earliest year in which the cycle has sunspot data. Plot μ₀ versus peak amplitude and fit a curve. Does a stronger cycle start at a higher latitude?

2. **Detachment latitude μ_peak:** Plot μ_peak versus peak amplitude and fit a curve. Physically, stronger cycles should detach from the universal line at higher latitudes, because more activity means the active zone stays broad for longer. Do the data support this?

3. **Poleward slope m_i:** Plot m_i versus peak amplitude and fit a curve. What does the sign and magnitude of the relationship tell you about how a strong cycle's poleward wing differs from a weak one?

Start with a linear fit for each relationship. If the scatter looks non-linear, try a power law or an exponential.

In [ ]:
# Task 19: Relating Cycle Amplitude to Wing Shape Parameters

def linear_fit(x, a, b):
    return a * x + b

# Build (cycle, hemisphere) → peak amplitude lookup from Task 18
amp_lookup = {
    (row["cycle"], row["hemisphere"]): row["peak_amplitude"]
    for _, row in peaks_df.iterrows()
}

# Assemble: shape params from Section 5  +  amplitude from Task 18
# μ₀ = value of the universal mean path at the *minimum* τ_refined observed
# for that hemisphere-cycle — the latitude where the cycle first shows activity.
records_19 = []
for r in results_pl:
    cyc, hemi = r["cycle"], r["hemisphere"]
    amp = amp_lookup.get((int(cyc), hemi))
    if amp is None or np.isnan(amp):
        continue
    mask = (df["CYCLE"] == cyc) & (df["hemisphere"] == hemi) & df["tau_refined"].notna()
    df_sel = df[mask]
    if len(df_sel) == 0:
        continue
    tau_start = df_sel["tau_refined"].min()
    mu0 = float(exp_decay(tau_start, a_mu_univ, b_mu_univ))
    records_19.append(dict(
        cycle=int(cyc), hemisphere=hemi,
        amplitude=float(amp), mu0=mu0,
        mu_peak=float(r["mu_peak"]),
        m_i=float(r["m_i"]),
    ))

df19 = pd.DataFrame(records_19)
print(df19.to_string(index=False))

# ── Scatter plots + linear fits ───────────────────────────────────────────
param_info = [
    ("mu0",     r"$\mu_0$ — earliest mean latitude (°)",    "tab:purple"),
    ("mu_peak", r"$\mu_{\rm peak}$ — detachment latitude (°)", "tab:orange"),
    ("m_i",     r"$m_i$ — poleward slope",                   "tab:green"),
]

fit_results_19 = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (col, ylabel, color) in zip(axes, param_info):
    vals = df19[["amplitude", col]].dropna()
    x, y = vals["amplitude"].values, vals[col].values
    popt, _ = curve_fit(linear_fit, x, y, p0=[0.0, float(np.mean(y))])
    a_fit, b_fit = popt
    fit_results_19[col] = (a_fit, b_fit)

    amp_grid_19 = np.linspace(x.min() * 0.9, x.max() * 1.05, 200)
    ax.scatter(x, y, color=color, s=40, alpha=0.80, edgecolors="none", zorder=3)
    ax.plot(amp_grid_19, linear_fit(amp_grid_19, a_fit, b_fit),
            color="black", linewidth=2,
            label=f"y = {a_fit:.5f}·A + {b_fit:.2f}")

    for _, row in df19.iterrows():
        if pd.notna(row["amplitude"]) and pd.notna(row[col]):
            ax.annotate(f"{int(row['cycle'])}{row['hemisphere'][0].upper()}",
                        (row["amplitude"], row[col]),
                        fontsize=5.5, alpha=0.55, xytext=(2, 2),
                        textcoords="offset points")

    ax.set_xlabel("Peak amplitude (MSH)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{col} vs amplitude")
    ax.legend(fontsize=8)

plt.suptitle("Task 19 — Wing shape parameters vs cycle peak amplitude", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print("\nFitted linear relationships:")
for col, (a, b) in fit_results_19.items():
    print(f"  {col:8s} = {a:+.6f} · A  +  {b:.3f}")

print("\nPhysical interpretation:")
print("  μ₀    vs A: positive slope — stronger cycles begin at higher latitudes (Waldmeier effect).")
print("  μ_peak vs A: positive slope — stronger cycles stay broad longer before collapsing.")
print("  m_i   vs A: the sign and magnitude tell us how steeply the poleward wing descends;")
print("               a more negative slope in a strong cycle means a broader, taller peak.")

## Task 20 — Putting It All Together: Synthesizing a Wing from Amplitude

This is the payoff task. Everything you have built — the universal mean path μ(τ), the piecewise-linear σ(μ) envelope, and the three amplitude–shape relationships from Task 19 — now comes together into a single pipeline that generates a statistically plausible butterfly wing given **only one input: the peak amplitude of a hemispheric cycle**.

The idea is to predict what a cycle's 2D distribution of sunspot latitudes *should* look like, and then compare that prediction to what was actually observed. Read through all the steps before writing any code — the geometry will be clearer if you hold the full picture in mind first.

1. Pick a hemispheric cycle from the dataset (e.g., cycle 24, northern hemisphere). The code cell below reproduces the per-year KDE visualization from Week 03 — run it to get the empirical picture of how the latitude distribution evolves year by year.

In [ ]:
# Task 8: KDE evolution through a solar cycle — butterfly diagram + per-year KDE profiles
import matplotlib.dates as mdates
from scipy.stats import gaussian_kde

cycle_number = 16    # Change to explore other cycles
hemisphere   = "south"  # Try "south" — Spörer's Law holds in both hemispheres

# Filter to cycle + hemisphere
mask = (df["CYCLE"] == cycle_number) & (df["hemisphere"] == hemisphere)
df_cyc_hemi = df[mask].copy()

years_in_cycle = sorted(df_cyc_hemi["year"].unique())
n_years = len(years_in_cycle)

# Colour map: early years → purple, late years → yellow (viridis is perceptually uniform
# and appropriate for a strictly increasing quantity like time)
cmap = plt.get_cmap("viridis", n_years)

# Latitude grid for evaluating KDEs (stay within the Spörer zone)
lat_grid = np.linspace(0, 45, 300) if hemisphere == "north" else np.linspace(-45, 0, 300)

# How wide (in days) should the tallest KDE peak be?
# ~250 days ≈ 2/3 of a year — wide enough to read, narrow enough not to overlap badly
kde_width_days = 250

fig, ax = plt.subplots(figsize=(12, 5))

# --- Background: butterfly diagram scatter ---
scatter_color = "tab:red" if hemisphere == "north" else "tab:blue"
ax.scatter(df_cyc_hemi["date"], df_cyc_hemi["latitude"],
           s=2, color=scatter_color, alpha=0.25, zorder=1, label="Sunspot groups")

# --- Foreground: per-year KDE profiles drawn vertically ---
for i, yr in enumerate(years_in_cycle):
    yr_lats = df_cyc_hemi.loc[df_cyc_hemi["year"] == yr, "latitude"].values
    if len(yr_lats) < 5:          # skip years with too few observations
        continue

    kde = gaussian_kde(yr_lats, bw_method=0.3)
    kde_vals = kde(lat_grid)

    # Normalise so the peak reaches kde_width_days on the x-axis
    kde_scaled = kde_vals / kde_vals.max() * kde_width_days

    # Anchor each profile at July 1 of that year
    center = pd.Timestamp(f"{int(yr)}-07-01")
    x_curve  = [center + pd.Timedelta(days=float(v)) for v in kde_scaled]
    x_anchor = [center] * len(lat_grid)

    color = cmap(i)
    ax.plot(x_curve, lat_grid, color=color, linewidth=1.8, alpha=0.9, zorder=3)
    ax.fill_betweenx(lat_grid, x_anchor, x_curve,
                     color=color, alpha=0.20, zorder=2)
    # Thin vertical baseline at the anchor date
    ax.axvline(center, color=color, linewidth=0.5, alpha=0.4, zorder=1)

    # Horizontal line at the yearly median latitude, extending to the KDE curve edge
    median_lat = np.median(yr_lats)
    kde_at_median = float(kde(np.array([median_lat]))[0])
    kde_at_median_scaled = kde_at_median / kde_vals.max() * kde_width_days
    x_median_end = center + pd.Timedelta(days=kde_at_median_scaled)
    ax.plot([center, x_median_end], [median_lat, median_lat],
            color=color, linewidth=2, linestyle="--", alpha=1.0, zorder=4)

# --- Colour bar (time axis) ---
sm = plt.cm.ScalarMappable(cmap="viridis",
                            norm=plt.Normalize(vmin=years_in_cycle[0],
                                               vmax=years_in_cycle[-1]))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("Year")

ax.set_title(
    f"Cycle {cycle_number} ({hemisphere}) — butterfly diagram with yearly KDE profiles\n"
    f"KDEs drift equatorward over time → Spörer's Law  |  dashed line = yearly median"
)
ax.set_xlabel("Date")
ax.set_ylabel("Latitude (degrees)")
ax.set_ylim((0, 45) if hemisphere == "north" else (-45, 0))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

# --- Numeric summary: KDE peak latitude per year (tracks Spörer drift) ---
print(f"Cycle {cycle_number} ({hemisphere}) — yearly median and KDE peak latitudes:")
for i, yr in enumerate(years_in_cycle):
    yr_lats = df_cyc_hemi.loc[df_cyc_hemi["year"] == yr, "latitude"].values
    if len(yr_lats) < 5:
        continue
    kde = gaussian_kde(yr_lats, bw_method=0.3)
    peak_lat   = lat_grid[np.argmax(kde(lat_grid))]
    median_lat = np.median(yr_lats)
    print(f"  {int(yr)}: median = {median_lat:.1f}°   KDE peak = {peak_lat:.1f}°")

2. Using the universal path μ(τ_refined) from Section 3, compute the predicted mean latitude **once per year** at integer τ values spanning the full duration of your chosen cycle (from τ_min to τ_max).

3. Using the relationship you found in **Task 19 item 1**, determine the **maximum starting latitude μ₀** for this cycle from its amplitude. Drop any annual latitude values above μ₀ — they correspond to years before significant sunspot activity began.

4. Using the relationship from **Task 19 item 2**, determine the **detachment latitude μ_peak** for this cycle from its amplitude.

5. Using the relationship from **Task 19 item 3**, determine the **poleward slope m_i** for this cycle from its amplitude.

6. For each surviving annual mean latitude μ(τ), evaluate the piecewise-linear model from Section 5 with the cycle-specific μ_peak and m_i you just derived. This gives you a spread σ(μ) for each year — a (mean, width) pair describing that year's slice of the butterfly wing.

7. Plot one Gaussian per year — centred at the predicted mean latitude, with width σ — in the same style as the KDE plot above (profiles drawn vertically, anchored at July 1 of each year, coloured by time). Overlay both the synthetic Gaussians and the real KDE profiles on the same axes.

**Reflection:** Where does the model agree with the data? Where does it fail, and why? What physical effects — cycle overlap, hemispheric asymmetry, early-cycle scatter — are absent from this simple parametric description?

In [ ]:
# Task 20: Putting It All Together — Synthesizing a Wing from Amplitude
# cycle_number and hemisphere are set in the cell above (default: 24 / north)

# ── Step 2: predict shape parameters from amplitude ───────────────────────
amp_20 = amp_lookup.get((cycle_number, hemisphere))
print(f"Cycle {cycle_number} ({hemisphere}): peak amplitude = {amp_20:.0f} MSH")

mu0_pred    = linear_fit(amp_20, *fit_results_19["mu0"])
mupeak_pred = linear_fit(amp_20, *fit_results_19["mu_peak"])
mi_pred     = linear_fit(amp_20, *fit_results_19["m_i"])
print(f"  Predicted μ₀     = {mu0_pred:.2f}°   (earliest significant emergence latitude)")
print(f"  Predicted μ_peak = {mupeak_pred:.2f}°   (detachment latitude)")
print(f"  Predicted m_i    = {mi_pred:.4f}    (poleward slope)")

# ── Step 3: annual mean latitudes from the universal path ─────────────────
t0_ref_20 = t0_refined[(cycle_number, hemisphere)]
df_sel_20  = df[(df["CYCLE"] == cycle_number) & (df["hemisphere"] == hemisphere)
                & df["tau_refined"].notna()]
tau_min_20 = df_sel_20["tau_refined"].min()
tau_max_20 = df_sel_20["tau_refined"].max()

# Use the same years as the KDE plot — guarantees 1:1 correspondence
year_ann = np.array(years_in_cycle)
tau_ann  = (year_ann + 0.5) - t0_ref_20
mu_ann   = exp_decay(tau_ann, a_mu_univ, b_mu_univ)
print(f"\nSynthetic years: {year_ann}  (μ range: {mu_ann.min():.1f}°–{mu_ann.max():.1f}°)")

# ── Steps 4–6: σ per year from the piecewise-linear model ────────────────
sigma_ann = np.array([
    piecewise_linear_wing(mu, m_shared_fit, b_shared_fit, mupeak_pred, mi_pred)
    for mu in mu_ann
])

# ── Step 7: overlay synthetic Gaussians on the real KDE butterfly diagram ─
sign     = +1 if hemisphere == "north" else -1
cmap_syn = plt.get_cmap("plasma", len(year_ann))

fig, ax = plt.subplots(figsize=(14, 6))

# Background scatter
sc_color = "tab:red" if hemisphere == "north" else "tab:blue"
ax.scatter(df_cyc_hemi["date"], df_cyc_hemi["latitude"],
           s=2, color=sc_color, alpha=0.15, zorder=1)

# Real KDE profiles (viridis — redrawn from the cell above)
for i, yr in enumerate(years_in_cycle):
    yr_lats = df_cyc_hemi.loc[df_cyc_hemi["year"] == yr, "latitude"].values
    if len(yr_lats) < 5:
        continue
    kde_obj  = gaussian_kde(yr_lats, bw_method=0.3)
    kde_vals = kde_obj(lat_grid)
    kde_sc   = kde_vals / kde_vals.max() * kde_width_days
    center   = pd.Timestamp(f"{int(yr)}-07-01")
    x_curve  = [center + pd.Timedelta(days=float(v)) for v in kde_sc]
    col_r    = cmap(i)   # viridis from the cell above
    ax.plot(x_curve, lat_grid, color=col_r, linewidth=1.5, alpha=0.65, zorder=3)
    ax.fill_betweenx(lat_grid, [center] * len(lat_grid), x_curve,
                     color=col_r, alpha=0.12, zorder=2)

# Synthetic Gaussian profiles (plasma, dashed)
for j, (yr, mu, sigma) in enumerate(zip(year_ann, mu_ann, sigma_ann)):
    if sigma <= 0:
        continue
    center_lat = sign * mu   # positive for north, negative for south
    gauss_vals = sp_norm.pdf(lat_grid, loc=center_lat, scale=sigma)
    if gauss_vals.max() == 0:
        continue
    gauss_sc = gauss_vals / gauss_vals.max() * kde_width_days
    center   = pd.Timestamp(f"{int(yr)}-07-01")
    x_curve  = [center + pd.Timedelta(days=float(v)) for v in gauss_sc]
    col_s    = cmap_syn(j)
    ax.plot(x_curve, lat_grid, color=col_s, linewidth=2.2, linestyle="--",
            alpha=0.95, zorder=5)
    ax.fill_betweenx(lat_grid, [center] * len(lat_grid), x_curve,
                     color=col_s, alpha=0.10, zorder=4)

# Colour bar for real KDE (viridis)
sm_r = plt.cm.ScalarMappable(cmap="viridis",
    norm=plt.Normalize(vmin=years_in_cycle[0], vmax=years_in_cycle[-1]))
sm_r.set_array([])
cbar = fig.colorbar(sm_r, ax=ax, pad=0.02)
cbar.set_label("Year (real KDE)")

# Legend
leg_elements = [
    Line2D([0], [0], color="gray",  linewidth=1.5, alpha=0.7,
           label="Real KDE (viridis)"),
    Line2D([0], [0], color="black", linewidth=2.2, linestyle="--",
           label="Synthetic Gaussian (plasma)"),
    Line2D([0], [0], color=sc_color, marker="o", markersize=4,
           linestyle="None", alpha=0.4, label="Observed sunspot groups"),
]
ax.legend(handles=leg_elements, loc="upper right", fontsize=9)

ax.set_title(
    f"Cycle {cycle_number} ({hemisphere}) — Real KDE vs Synthetic Gaussian\n"
    f"Amplitude = {amp_20:.0f} MSH  →  μ₀ = {mu0_pred:.1f}°   "
    f"μ_peak = {mupeak_pred:.1f}°   m_i = {mi_pred:.3f}"
)
ax.set_xlabel("Date")
ax.set_ylabel("Latitude (°)")
ax.set_ylim((0, 45) if hemisphere == "north" else (-45, 0))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()


---
## Setup: `compute_global_nll` — A Single Number to Rule Them All

Before the per-cycle diagnostics, run the cell below once. It does two things:

1. **Builds a compact data cache** (`cycle_data_global`) — a list of pre-computed (τ, μ, latitudes) triples for every valid year in every hemisphere-cycle. This is computed once and reused throughout Tasks 21–24, keeping DataFrame access out of any inner optimization loop.

2. **Defines `compute_global_nll(fit_results)`** — a function that takes a set of amplitude–shape coefficients and returns the **mean per-year-normalized NLL across the entire dataset**:

$$\text{Global NLL}(a_0, b_0, \ldots) = \frac{1}{N_{\text{terms}}} \sum_{\text{cycles}} \sum_{\text{years}} \left[ -\frac{1}{n_t} \sum_i \log \mathcal{N}(x_i \mid \mu_t, \sigma_t) \right]$$

This is your **scoreboard metric** for the rest of the notebook. Lower is better. The Task 19 two-stage fits give you the baseline; your job is to beat it.

```python
nll = compute_global_nll(fit_results_19)          # evaluate Task 19 baseline
nll = compute_global_nll(my_fit)                  # evaluate your model
nll = compute_global_nll(my_fit, m_sh=0.12, b_sh=-0.5)   # also optimize the equatorial line
```

In [ ]:
# Setup: pre-compute cycle data cache and define the global scoreboard metric.
# Run this cell once before executing Tasks 21–24.

df19_records_global = []
cycle_data_global   = []
for rec in df19.to_dict("records"):
    cyc, hemi = rec["cycle"], rec["hemisphere"]
    A = amp_lookup.get((int(cyc), hemi))
    if A is None:
        continue
    df_ch = df[(df["CYCLE"] == cyc) & (df["hemisphere"] == hemi)]
    years_data = []
    for yr in sorted(df_ch["year"].unique()):
        tau  = (yr + 0.5) - t0_refined[(int(cyc), hemi)]
        mu   = exp_decay(tau, a_mu_univ, b_mu_univ)
        lats = df_ch.loc[df_ch["year"] == yr, "latitude"].abs().values
        if len(lats) >= 5:
            years_data.append((tau, mu, lats))
    if years_data:
        df19_records_global.append(rec)
        cycle_data_global.append((A, years_data))


def compute_global_nll(fit_results, m_sh=None, b_sh=None):
    """
    Mean per-year-normalized NLL across all hemisphere-cycles — the scoreboard metric.

    Parameters
    ----------
    fit_results : dict with keys 'mu0', 'mu_peak', 'm_i'
        Each value is a (slope, intercept) tuple: param(A) = slope * A + intercept.
        Same format as fit_results_19 from Task 19.
    m_sh, b_sh : float, optional
        Override the universal equatorial line parameters.
        Defaults to m_shared_fit / b_shared_fit from Section 5.

    Returns
    -------
    float — lower is better.  Units: nats/year.
    """
    if m_sh is None: m_sh = m_shared_fit
    if b_sh is None: b_sh = b_shared_fit
    a_mu0,    b_mu0    = fit_results["mu0"]
    a_mupeak, b_mupeak = fit_results["mu_peak"]
    a_mi,     b_mi     = fit_results["m_i"]
    total, n_terms = 0.0, 0
    for A, years_data in cycle_data_global:
        mu0_p    = a_mu0 * A + b_mu0
        mupeak_p = a_mupeak * A + b_mupeak
        mi_p     = a_mi * A + b_mi
        for tau, mu, lats in years_data:
            if mu > mu0_p:
                continue
            sigma = piecewise_linear_wing(mu, m_sh, b_sh, mupeak_p, mi_p)
            if sigma <= 0:
                continue
            total -= sp_norm.logpdf(lats, loc=mu, scale=sigma).mean()
            n_terms += 1
    return total / max(n_terms, 1)


nll_baseline = compute_global_nll(fit_results_19)
n_cycles = len(cycle_data_global)
n_terms  = sum(len(yd) for _, yd in cycle_data_global)
print(f"Dataset: {n_cycles} hemisphere-cycles  |  {n_terms} valid year-cycle terms")
print(f"\nBaseline NLL (Task 19 two-stage fits): {nll_baseline:.5f} nats/year")
print("Beat this number in Tasks 23–24 and in the 'Going Further' experiments.")

---
# Week 07 Tasks: Building the Diffusion Forward Pass

Everything above this line is **recap and setup** — the classical statistical
model from Weeks 03–06, the per-cycle shape fits, and the global scoreboard
metric. Below this line begins the Week 07 work proper: turn the classical
model into a dataset of residuals, define the splits, and implement the
forward diffusion process in pure numpy.

The tasks build sequentially. Tasks 26–27 prepare the data; Tasks 28–29
implement the forward process; Tasks 30–32 verify it behaves the way the
mathematics says it should.


---
## Task 26 — Build the per-window dataset table

The training data for the diffusion model is a table where each row is one
**6-month window** of one hemispheric cycle, and the most important columns
are the empirical and parametric latitude histograms for that window. The
**residual** for the window is then *empirical histogram − parametric
histogram*, both expressed as densities on the same latitude grid.

Two design choices to flag explicitly before you start, because they affect
every downstream task:

- **Non-overlapping 6-month windows.** Each calendar half-year contributes one
  row. An alternative would be sliding 6-month windows that step by, say,
  one month — that gives more rows but they are not statistically
  independent. We choose non-overlapping windows so that "row count" and
  "independent observation count" are the same number.
- **15 latitude bins of 3° width covering 0°–45°.** The Spörer zone is
  bounded between roughly 5° and 40°, so bins 0 (0°–3°) and 14 (42°–45°)
  will be near-empty for most cycles. We keep them anyway: the residual at
  those bins is small *by construction*, and the diffusion model will learn
  this trivial structure quickly. Trimming the grid would create a special
  case that is more confusing than useful.

**Tasks:**
- For every (cycle, hemisphere) with `cycle ≥ 12`, identify the calendar
  span of the cycle and partition it into consecutive non-overlapping
  6-month windows. Discard any window with fewer than 20 observed sunspot
  group emergences (too few to form a meaningful histogram).
- For each retained window, compute and store as columns:
  - `cycle`, `hemisphere`.
  - `amplitude` — the **hemispheric cycle peak amplitude** from the
    Task 18 lookup. This is a **cycle-level constant**: every window of the
    same hemispheric cycle gets the same amplitude.
  - `tau_center` — the value of τ_refined at the centre of the 6-month window.
  - `mu_universal` — the universal-path mean latitude at `tau_center`,
    computed from `exp_decay(tau_center, a_mu_univ, b_mu_univ)`.
  - `area_smoothed` — the 12-month-smoothed total hemispheric area at the
    window centre, sampled from `smooth_north` / `smooth_south`.
  - 15 columns `hist_emp_00` … `hist_emp_14` — the **empirical density**
    of `abs_latitude` on the bin grid `np.linspace(0, 45, 16)`. Use
    `np.histogram(..., density=True)`.
  - 15 columns `hist_par_00` … `hist_par_14` — the **parametric density**
    on the same bin grid, computed by **integrating the per-window Gaussian
    over each bin** (`scipy.stats.norm.cdf` at the bin edges, then divide by
    bin width to convert to density). The Gaussian's parameters
    (μ_t, σ_t) come from the per-cycle shape fit in `results_pl`:
    μ_t = `exp_decay(tau_center, a_mu_univ, b_mu_univ)` and σ_t from
    `piecewise_linear_wing(μ_t, m_shared_fit, b_shared_fit, μ_peak_cycle, m_i_cycle)`.

**Important:** evaluating the Gaussian at bin centers is *not* the same as
integrating it over each bin. For 3° bins and σ in the 3°–7° range, the
two differ by a few percent. Integration is the principled choice and is
what we use here, so the residual is a clean difference of two density
objects.

- **Sanity plot:** pick one row of the table at random and overplot the
  empirical histogram and the parametric histogram on the same axes (bar
  chart for empirical, line plot or bar chart for parametric). They should
  resemble each other but not be identical — the difference between them
  is exactly what the diffusion model will learn to generate.


In [ ]:
# Put your code here for Task 26.
# Task 26: Build the per-window dataset table
# Depends on: df, t0_refined, a_mu_univ, b_mu_univ, exp_decay,
#             m_shared_fit, b_shared_fit, piecewise_linear_wing,
#             results_pl, amp_lookup, smooth_north, smooth_south

from scipy.stats import norm as sp_norm

# ── Grid definition ───────────────────────────────────────────────────────
N_BINS      = 15
LAT_EDGES   = np.linspace(0, 45, N_BINS + 1)   # 16 edges → 15 bins of 3° each
LAT_CENTERS = 0.5 * (LAT_EDGES[:-1] + LAT_EDGES[1:])
BIN_WIDTH   = LAT_EDGES[1] - LAT_EDGES[0]       # 3.0°
MIN_COUNTS  = 20                                 # discard windows below this

# ── Build per-cycle shape parameter lookup from results_pl ────────────────
# Each (cycle, hemisphere) gets its fitted mu_peak and m_i
shape_lookup = {
    (int(r["cycle"]), r["hemisphere"]): (r["mu_peak"], r["m_i"])
    for r in results_pl
}

# ── Main loop: iterate over every (cycle, hemisphere) ─────────────────────
rows = []

for cyc in cycles_13:
    for hemi in ["north", "south"]:

        # Need refined t0 and shape parameters
        if (cyc, hemi) not in t0_refined:
            continue
        if (int(cyc), hemi) not in shape_lookup:
            continue

        amplitude = amp_lookup.get((int(cyc), hemi))
        if amplitude is None:
            continue

        mu_peak_cyc, m_i_cyc = shape_lookup[(int(cyc), hemi)]
        t0_ref = t0_refined[(cyc, hemi)]
        smooth_hemi = smooth_north if hemi == "north" else smooth_south

        # All sunspot groups for this hemisphere-cycle
        df_ch = df[(df["CYCLE"] == cyc) & (df["hemisphere"] == hemi)].copy()
        if len(df_ch) == 0:
            continue

        # Calendar span of this cycle
        date_min = df_ch["date"].min()
        date_max = df_ch["date"].max()

        # Generate non-overlapping 6-month window boundaries
        # Anchor at Jan 1 of the first year, step by 6 months
        window_start = pd.Timestamp(f"{date_min.year}-01-01")
        window_ends  = []
        while window_start <= date_max:
            window_end = window_start + pd.DateOffset(months=6)
            window_ends.append((window_start, window_end))
            window_start = window_end

        for w_start, w_end in window_ends:
            # Observations in this window
            mask_win = (df_ch["date"] >= w_start) & (df_ch["date"] < w_end)
            df_win   = df_ch[mask_win]
            n_obs    = len(df_win)

            if n_obs < MIN_COUNTS:
                continue

            # Window center date and tau
            center_date = w_start + (w_end - w_start) / 2
            decimal_center = (center_date.year
                              + center_date.timetuple().tm_yday / 365.25)
            tau_center = decimal_center - t0_ref

            # Universal path mean latitude at window center
            mu_universal = float(exp_decay(tau_center, a_mu_univ, b_mu_univ))

            # Smoothed area at window center
            idx_smooth = smooth_hemi.index.get_indexer(
                [center_date], method="nearest")[0]
            area_smoothed = float(smooth_hemi.iloc[idx_smooth]) \
                            if idx_smooth >= 0 else np.nan

            # ── Empirical histogram ───────────────────────────────────────
            abs_lats = df_win["abs_latitude"].values
            hist_emp, _ = np.histogram(abs_lats, bins=LAT_EDGES, density=True)

            # ── Parametric histogram (integrated Gaussian) ────────────────
            sigma_par = piecewise_linear_wing(
                mu_universal, m_shared_fit, b_shared_fit,
                mu_peak_cyc, m_i_cyc)

            if sigma_par <= 0:
                continue

            # Integrate N(mu_universal, sigma_par) over each bin
            cdf_vals  = sp_norm.cdf(LAT_EDGES, loc=mu_universal, scale=sigma_par)
            bin_probs = np.diff(cdf_vals)          # probability mass per bin
            # Normalize to density (probability mass / bin width)
            # Also renormalize so the histogram sums to 1 over the 0–45° grid
            total_prob = bin_probs.sum()
            if total_prob <= 0:
                continue
            hist_par = (bin_probs / total_prob) / BIN_WIDTH

            # ── Assemble row ──────────────────────────────────────────────
            row = dict(
                cycle         = int(cyc),
                hemisphere    = hemi,
                amplitude     = float(amplitude),
                tau_center    = float(tau_center),
                mu_universal  = float(mu_universal),
                area_smoothed = float(area_smoothed) if not np.isnan(area_smoothed) else np.nan,
                n_obs         = int(n_obs),
                window_start  = w_start,
                window_end    = w_end,
            )
            for k in range(N_BINS):
                row[f"hist_emp_{k:02d}"] = float(hist_emp[k])
                row[f"hist_par_{k:02d}"] = float(hist_par[k])

            rows.append(row)

df_windows = pd.DataFrame(rows).reset_index(drop=True)

# ── Summary ───────────────────────────────────────────────────────────────
emp_cols = [f"hist_emp_{k:02d}" for k in range(N_BINS)]
par_cols = [f"hist_par_{k:02d}" for k in range(N_BINS)]
res_cols = [f"hist_res_{k:02d}" for k in range(N_BINS)]

# Add residual columns: empirical − parametric
for k in range(N_BINS):
    df_windows[f"hist_res_{k:02d}"] = (df_windows[f"hist_emp_{k:02d}"]
                                       - df_windows[f"hist_par_{k:02d}"])

print(f"Dataset shape   : {df_windows.shape}")
print(f"Total windows   : {len(df_windows)}")
print(f"Cycles covered  : {sorted(df_windows['cycle'].unique())}")
print(f"Windows per cycle (mean): {df_windows.groupby('cycle').size().mean():.1f}")
print(f"\nColumn groups:")
print(f"  Metadata : cycle, hemisphere, amplitude, tau_center, "
      f"mu_universal, area_smoothed, n_obs")
print(f"  Empirical: {emp_cols[0]} … {emp_cols[-1]}")
print(f"  Parametric:{par_cols[0]} … {par_cols[-1]}")
print(f"  Residual : {res_cols[0]} … {res_cols[-1]}")
print(f"\nFirst 5 rows (metadata only):")
print(df_windows[["cycle", "hemisphere", "amplitude", "tau_center",
                   "mu_universal", "area_smoothed", "n_obs"]].head())

# ── Sanity check: histogram sums ─────────────────────────────────────────
emp_sums = df_windows[emp_cols].sum(axis=1) * BIN_WIDTH
par_sums = df_windows[par_cols].sum(axis=1) * BIN_WIDTH
print(f"\nSanity check — histogram × bin_width should ≈ 1.0:")
print(f"  Empirical  : mean = {emp_sums.mean():.4f}  std = {emp_sums.std():.4f}")
print(f"  Parametric : mean = {par_sums.mean():.4f}  std = {par_sums.std():.4f}")

# ── Sanity plot: one random window ───────────────────────────────────────
rng_26  = np.random.default_rng(7)
idx_rnd = int(rng_26.integers(0, len(df_windows)))
row_rnd = df_windows.iloc[idx_rnd]

emp_vals = row_rnd[emp_cols].values.astype(float)
par_vals = row_rnd[par_cols].values.astype(float)
res_vals = row_rnd[res_cols].values.astype(float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: empirical vs parametric ---
ax = axes[0]
bar_width = BIN_WIDTH * 0.42
ax.bar(LAT_CENTERS - bar_width/2, emp_vals, width=bar_width,
       color="steelblue", alpha=0.75, label="Empirical")
ax.bar(LAT_CENTERS + bar_width/2, par_vals, width=bar_width,
       color="tomato",   alpha=0.75, label="Parametric")
ax.set_xlabel("Absolute latitude (°)")
ax.set_ylabel("Density (per degree)")
ax.set_title(
    f"Sanity plot — window {idx_rnd}\n"
    f"Cycle {int(row_rnd['cycle'])} {row_rnd['hemisphere']}  |  "
    f"τ = {row_rnd['tau_center']:.2f} yr  |  "
    f"μ = {row_rnd['mu_universal']:.1f}°  |  "
    f"n = {int(row_rnd['n_obs'])} groups"
)
ax.legend()

# --- Right: residual ---
ax2 = axes[1]
colors_res = ["tab:green" if v >= 0 else "tab:red" for v in res_vals]
ax2.bar(LAT_CENTERS, res_vals, width=BIN_WIDTH * 0.7,
        color=colors_res, alpha=0.8)
ax2.axhline(0, color="black", linewidth=1)
ax2.set_xlabel("Absolute latitude (°)")
ax2.set_ylabel("Residual density (empirical − parametric)")
ax2.set_title(
    "Residual histogram\n"
    "Green = more sunspots than model predicted  |  "
    "Red = fewer"
)

plt.tight_layout()
plt.show()

# ── Distribution of residuals across all windows ──────────────────────────
all_residuals = df_windows[res_cols].values.flatten()

fig2, axes2 = plt.subplots(1, 2, figsize=(13, 4))

# Residual distribution
axes2[0].hist(all_residuals, bins=60, color="slateblue", alpha=0.8, density=True)
axes2[0].axvline(0, color="black", linewidth=1)
axes2[0].set_xlabel("Residual density value")
axes2[0].set_ylabel("Probability density")
axes2[0].set_title(
    f"Distribution of all residual values\n"
    f"mean = {all_residuals.mean():.4f}  "
    f"std = {all_residuals.std():.4f}  "
    f"(should be near-zero mean)"
)

# Mean residual profile across all windows
mean_res = df_windows[res_cols].mean().values
std_res  = df_windows[res_cols].std().values
axes2[1].bar(LAT_CENTERS, mean_res, width=BIN_WIDTH * 0.7,
             color=["tab:green" if v >= 0 else "tab:red" for v in mean_res],
             alpha=0.8)
axes2[1].fill_between(LAT_CENTERS,
                       mean_res - std_res, mean_res + std_res,
                       alpha=0.2, color="gray", label="±1 std")
axes2[1].axhline(0, color="black", linewidth=1)
axes2[1].set_xlabel("Absolute latitude (°)")
axes2[1].set_ylabel("Mean residual density")
axes2[1].set_title(
    "Mean residual profile across all windows\n"
    "Systematic non-zero bins = parametric model has a consistent bias here"
)
axes2[1].legend()

plt.tight_layout()
plt.show()

print(f"\ndf_windows ready — {len(df_windows)} rows × {len(df_windows.columns)} columns")
print("Key columns for diffusion model training:")
print("  Conditioning : amplitude, tau_center, mu_universal, area_smoothed")
print("  Target       : hist_res_00 … hist_res_14  (what the model must learn)")
print("  Reference    : hist_par_00 … hist_par_14  (added back at inference time)")


---
## Task 27 — Stratified train / validation / test split

Diffusion training, like supervised training, needs three disjoint splits.
The roles, however, are slightly different from what you may be used to:

- **Training set:** what the model sees during gradient updates.
- **Validation set:** used for **early stopping** (decide when to halt
  training) and for **schedule / hyperparameter selection** (compare
  candidate noise schedules, network widths, weighting schemes). Touched
  many times during model development, but never used in gradient updates.
- **Test set:** touched only when reporting the final model's performance.
  If you tune anything against the test set, your reported numbers are
  optimistic and will not reproduce.

The split is by **hemispheric cycle**, not by row. Two different windows
of the same hemispheric cycle are not independent observations: they share
the same amplitude, the same per-cycle shape parameters, and overlapping
physics. Splitting by row would leak information from train to validation
and inflate your scores.

We further want the split to be **stratified by cycle amplitude** so that
all three sets contain a representative mix of strong and weak cycles. A
random split could land all the weakest cycles in the test set and produce
a useless evaluation.

**Tasks:**
- Build a table with one row per hemispheric cycle, columns
  `(cycle, hemisphere, amplitude)`, where `amplitude` is the same Task 18
  peak amplitude used in Task 26.
- Sort the table by amplitude, **strongest first**.
- Walk down the sorted list and assign every fifth cycle (starting from
  index 0) to the **test set** — this is uniform sampling along the
  amplitude axis, which is what stratification means in this context.
- From the remaining cycles, walk down the sorted list and assign every
  fifth cycle (starting from index 0 of the remainder) to the
  **validation set**.
- Everything left is the **training set**.
- **Sanity plot:** plot a histogram (or empirical CDF) of cycle amplitudes
  for each of the three sets on the same axes. The three distributions
  should look similar — same range, same overall shape. If one set is
  systematically biased toward strong or weak cycles, the stratification
  has gone wrong.

**Ramification:** with ~46 hemispheric cycles total, a 60/20/20 split
yields roughly 28 training, 9 validation, 9 test cycles. This is decisively
small-data territory. Cross-validation across the train/validation
boundary would give more statistically robust hyperparameter choices, and
is a natural Week 09 extension once the basic pipeline works.


In [ ]:
# Put your code here for Task 27.
# Task 27: Stratified train / validation / test split
# Depends on: df_windows (from Task 26), amp_lookup, cycles_13

# ── Step 1: one row per hemispheric cycle ─────────────────────────────────
hc_records = []
for cyc in cycles_13:
    for hemi in ["north", "south"]:
        amp = amp_lookup.get((int(cyc), hemi))
        if amp is None:
            continue
        # Only include cycles that actually appear in df_windows
        n_windows = len(df_windows[
            (df_windows["cycle"] == int(cyc)) &
            (df_windows["hemisphere"] == hemi)
        ])
        if n_windows == 0:
            continue
        hc_records.append(dict(
            cycle      = int(cyc),
            hemisphere = hemi,
            amplitude  = float(amp),
            n_windows  = n_windows,
        ))

df_hc = (pd.DataFrame(hc_records)
           .sort_values("amplitude", ascending=False)
           .reset_index(drop=True))

print(f"Total hemispheric cycles in dataset: {len(df_hc)}")
print(f"Total windows in df_windows        : {len(df_windows)}")
print(f"\nAmplitude range: {df_hc['amplitude'].min():.0f} – "
      f"{df_hc['amplitude'].max():.0f} MSH")
print(f"\nSorted hemispheric cycles (strongest → weakest):")
print(df_hc[["cycle", "hemisphere", "amplitude", "n_windows"]].to_string(index=True))

# ── Step 2: stratified split by every-5th sampling ───────────────────────
# Walk the amplitude-sorted list:
#   indices 0, 5, 10, 15, ... → test
#   of the remainder, indices 0, 5, 10, ... → validation
#   everything left → training

test_mask = np.zeros(len(df_hc), dtype=bool)
test_mask[::5] = True

df_test      = df_hc[test_mask].copy()
df_remaining = df_hc[~test_mask].reset_index(drop=True)

val_mask = np.zeros(len(df_remaining), dtype=bool)
val_mask[::5] = True

df_val   = df_remaining[val_mask].copy()
df_train = df_remaining[~val_mask].copy()

# Assign split labels back to df_hc for reference
split_map = {}
for _, row in df_train.iterrows():
    split_map[(row["cycle"], row["hemisphere"])] = "train"
for _, row in df_val.iterrows():
    split_map[(row["cycle"], row["hemisphere"])] = "val"
for _, row in df_test.iterrows():
    split_map[(row["cycle"], row["hemisphere"])] = "test"

df_hc["split"] = df_hc.apply(
    lambda r: split_map.get((r["cycle"], r["hemisphere"]), "unknown"), axis=1)

# ── Step 3: propagate split labels to df_windows ─────────────────────────
df_windows["split"] = df_windows.apply(
    lambda r: split_map.get((r["cycle"], r["hemisphere"]), "unknown"), axis=1)

# ── Step 4: summary table ─────────────────────────────────────────────────
split_summary = []
for split_name, df_split in [("train", df_train), ("val", df_val), ("test", df_test)]:
    n_cyc  = len(df_split)
    n_win  = df_windows[df_windows["split"] == split_name].shape[0]
    amp_min = df_split["amplitude"].min()
    amp_max = df_split["amplitude"].max()
    amp_med = df_split["amplitude"].median()
    split_summary.append(dict(
        split=split_name, n_cycles=n_cyc, n_windows=n_win,
        amp_min=amp_min, amp_max=amp_max, amp_median=amp_med,
    ))

df_summary = pd.DataFrame(split_summary)
print("\n" + "=" * 70)
print("  Split summary")
print("=" * 70)
print(df_summary.to_string(index=False))
print("=" * 70)

print("\nCycles per split:")
for split_name in ["train", "val", "test"]:
    rows = df_hc[df_hc["split"] == split_name][
        ["cycle", "hemisphere", "amplitude"]].values
    cyc_strs = [f"{int(r[0])}{r[1][0].upper()}({r[2]:.0f})" for r in rows]
    print(f"  {split_name:<6}: {', '.join(cyc_strs)}")

# ── Step 5: convenience split DataFrames of df_windows ────────────────────
df_windows_train = df_windows[df_windows["split"] == "train"].reset_index(drop=True)
df_windows_val   = df_windows[df_windows["split"] == "val"  ].reset_index(drop=True)
df_windows_test  = df_windows[df_windows["split"] == "test" ].reset_index(drop=True)

print(f"\nWindow counts: train={len(df_windows_train)}  "
      f"val={len(df_windows_val)}  test={len(df_windows_test)}")

# ── Step 6: sanity plots ──────────────────────────────────────────────────
split_styles = {
    "train": ("tab:blue",   "Train",      0.55),
    "val":   ("tab:orange", "Validation", 0.70),
    "test":  ("tab:green",  "Test",       0.85),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: amplitude histogram per split ---
ax = axes[0]
amp_bins = np.linspace(df_hc["amplitude"].min() * 0.9,
                        df_hc["amplitude"].max() * 1.05, 12)
for split_name, (color, label, _) in split_styles.items():
    amps = df_hc.loc[df_hc["split"] == split_name, "amplitude"].values
    ax.hist(amps, bins=amp_bins, color=color, alpha=0.55,
            label=f"{label} (n={len(amps)})", edgecolor="white")

ax.set_xlabel("Peak amplitude (MSH)")
ax.set_ylabel("Number of hemispheric cycles")
ax.set_title("Amplitude distribution by split\n"
             "All three sets should span the full amplitude range")
ax.legend()

# --- Right: empirical CDF of amplitude per split ---
ax2 = axes[1]
for split_name, (color, label, _) in split_styles.items():
    amps = np.sort(df_hc.loc[df_hc["split"] == split_name,
                              "amplitude"].values)
    cdf  = np.arange(1, len(amps) + 1) / len(amps)
    ax2.step(amps, cdf, color=color, linewidth=2.5,
             label=f"{label} (n={len(amps)})", where="post")

# Reference: full dataset CDF
amps_all = np.sort(df_hc["amplitude"].values)
cdf_all  = np.arange(1, len(amps_all) + 1) / len(amps_all)
ax2.step(amps_all, cdf_all, color="black", linewidth=1.2,
         linestyle="--", alpha=0.5, label="All cycles", where="post")

ax2.set_xlabel("Peak amplitude (MSH)")
ax2.set_ylabel("Empirical CDF")
ax2.set_title("Empirical CDF of amplitude by split\n"
              "Curves should track each other closely — divergence = bad stratification")
ax2.legend()
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

# ── Step 7: amplitude sorted view — visualize which cycles go where ───────
fig2, ax3 = plt.subplots(figsize=(14, 4))

colors_map = {"train": "tab:blue", "val": "tab:orange", "test": "tab:green"}
for _, row in df_hc.iterrows():
    color = colors_map[row["split"]]
    ax3.bar(row.name, row["amplitude"],
            color=color, alpha=0.75, edgecolor="white", width=0.8)
    ax3.text(row.name, row["amplitude"] + 5,
             f"{int(row['cycle'])}{row['hemisphere'][0].upper()}",
             ha="center", va="bottom", fontsize=6.5, rotation=90)

# Legend
from matplotlib.patches import Patch
legend_els = [Patch(color=c, alpha=0.75, label=s.capitalize())
              for s, c in colors_map.items()]
ax3.legend(handles=legend_els, loc="upper right")
ax3.set_xlabel("Rank (strongest → weakest)")
ax3.set_ylabel("Peak amplitude (MSH)")
ax3.set_title("Stratified split — amplitude-sorted hemispheric cycles\n"
              "Every 5th cycle (blue=train, orange=val, green=test) "
              "sampled uniformly along the amplitude axis")
ax3.set_xticks([])
plt.tight_layout()
plt.show()

# ── Step 8: leakage check ─────────────────────────────────────────────────
# Verify no cycle appears in more than one split
all_assigned = [(r["cycle"], r["hemisphere"]) for _, r in df_hc.iterrows()]
unique_assigned = set(all_assigned)
assert len(all_assigned) == len(unique_assigned), "Duplicate cycle assignments!"

train_set = set(zip(df_train["cycle"], df_train["hemisphere"]))
val_set   = set(zip(df_val["cycle"],   df_val["hemisphere"]))
test_set  = set(zip(df_test["cycle"],  df_test["hemisphere"]))

assert len(train_set & val_set)  == 0, "Train/val overlap!"
assert len(train_set & test_set) == 0, "Train/test overlap!"
assert len(val_set   & test_set) == 0, "Val/test overlap!"

print("✓ Leakage check passed — no cycle appears in more than one split")
print(f"✓ df_windows has 'split' column: "
      f"{df_windows['split'].value_counts().to_dict()}")
print(f"\ndf_windows_train / df_windows_val / df_windows_test ready")
print("These are the inputs to the diffusion model training pipeline.")


---
## Task 28 — Define the cosine noise schedule

The forward process corrupts a clean residual r into a noisy version r_t at
**timestep** t ∈ {0, 1, …, T} according to

$$r_t \;=\; \alpha_t \cdot r \;+\; \sigma_t \cdot \varepsilon, \qquad
\varepsilon \sim \mathcal{N}(0, I).$$

The pair (α_t, σ_t) is the **noise schedule**. It is fixed before training
begins, computed once, stored as two arrays of length T+1, and looked up by
index thereafter.

We use the **cosine schedule** of Nichol & Dhariwal (2021), which is
**variance-preserving**: α_t² + σ_t² = 1 at every t, so the total variance
of r_t stays bounded as t increases. This contrasts with the
**variance-exploding** family (used in some image-generation papers) where
σ_t grows without bound while α_t = 1 throughout. For low-dimensional data
like ours, variance-preserving is more stable and easier to train.

The cosine schedule defines a smooth function

$$\bar{\alpha}_t \;=\; \frac{\cos\!\big(\tfrac{\pi}{2} \cdot \tfrac{t/T + s}{1 + s}\big)^2}{\cos\!\big(\tfrac{\pi}{2} \cdot \tfrac{s}{1 + s}\big)^2}$$

with a small offset `s ≈ 0.008`, and then

$$\alpha_t = \sqrt{\bar{\alpha}_t}, \qquad \sigma_t = \sqrt{1 - \bar{\alpha}_t}.$$

The **signal-to-noise ratio** at timestep t is

$$\mathrm{SNR}(t) \;=\; \frac{\alpha_t^2}{\sigma_t^2} \;=\; \frac{\bar{\alpha}_t}{1 - \bar{\alpha}_t}.$$

It starts very large at t = 0 (almost pure signal), passes through 1 at the
schedule's halfway point, and falls toward 0 at t = T (almost pure noise).
The **shape of the SNR descent** is what the schedule actually controls;
α_t and σ_t are bookkeeping derived from it.

**Tasks:**
- Set `T = 200`. Implement the cosine schedule above with offset
  `s = 0.008`. Compute `alpha_bar` (length T+1), `alpha` (length T+1), and
  `sigma` (length T+1) as numpy arrays.
- For numerical stability, **clip** the per-step noise variance — define
  `beta_t = 1 - alpha_bar[t] / alpha_bar[t-1]` and clip `beta_t` to
  `[1e-8, 0.999]`. Recompute `alpha_bar` from the clipped `beta` if
  needed; this prevents pathological behaviour at the extreme timesteps.
- Plot α_t, σ_t, and SNR(t) on the same figure (use a log y-axis for SNR).
  Mark the timestep at which SNR(t) = 1 with a vertical line.
- **Reading the plot:** identify the timesteps where (i) the signal is
  almost untouched (high SNR), (ii) the signal is roughly half-corrupted
  (SNR ≈ 1), (iii) the signal is essentially gone (low SNR). These three
  regimes correspond to the three regimes the diffusion model has to
  handle next week.


In [ ]:
# Put your code here for Task 28.
# Task 28: Define the cosine noise schedule
# Standalone — no dependencies beyond numpy and matplotlib

import numpy as np
import matplotlib.pyplot as plt

# ── Parameters ────────────────────────────────────────────────────────────
T = 200        # total diffusion timesteps
s = 0.008      # small offset to prevent β_0 from being exactly 0

# ── Step 1: cosine schedule for ᾱ_t ──────────────────────────────────────
# f(t) = cos²( (t/T + s) / (1 + s) · π/2 )
# ᾱ_t  = f(t) / f(0)

t_arr = np.arange(0, T + 1, dtype=np.float64)   # shape (T+1,)

def f_cos(t, T=T, s=s):
    return np.cos((t / T + s) / (1.0 + s) * np.pi / 2.0) ** 2

f_vals    = f_cos(t_arr)
alpha_bar = f_vals / f_vals[0]    # normalize so ᾱ_0 = 1 exactly

# ── Step 2: per-step beta + clipping for numerical stability ──────────────
# β_t = 1 - ᾱ_t / ᾱ_{t-1}   for t = 1, …, T
# β_0 is defined as 0 (no corruption at t=0)
beta = np.empty(T + 1)
beta[0] = 0.0
beta[1:] = 1.0 - alpha_bar[1:] / alpha_bar[:-1]

# Clip to prevent numerical pathologies at extreme timesteps
beta = np.clip(beta, 1e-8, 0.999)

# Recompute alpha_bar from the clipped beta to stay consistent
# ᾱ_t = ∏_{s=1}^{t} (1 - β_s)
alpha_bar_clipped       = np.empty(T + 1)
alpha_bar_clipped[0]    = 1.0
alpha_bar_clipped[1:]   = np.cumprod(1.0 - beta[1:])

# Overwrite with clipped version
alpha_bar = alpha_bar_clipped

# ── Step 3: derive α_t and σ_t ────────────────────────────────────────────
# Variance-preserving: α_t² + σ_t² = 1
# α_t here is sqrt(ᾱ_t)  — the signal scaling at timestep t
# σ_t = sqrt(1 - ᾱ_t)    — the noise scaling at timestep t
alpha = np.sqrt(alpha_bar)          # signal coefficient
sigma = np.sqrt(1.0 - alpha_bar)    # noise coefficient

# ── Step 4: signal-to-noise ratio ────────────────────────────────────────
# SNR(t) = ᾱ_t / (1 - ᾱ_t) = α_t² / σ_t²
# Clip denominator to avoid division by zero at t=T
snr = alpha_bar / np.maximum(1.0 - alpha_bar, 1e-10)

# ── Step 5: find SNR = 1 crossing ─────────────────────────────────────────
# SNR crosses 1 when ᾱ_t = 0.5
snr_one_idx = int(np.argmin(np.abs(snr - 1.0)))
snr_one_t   = t_arr[snr_one_idx]
print(f"SNR = 1 at t = {snr_one_t:.0f}  ({snr_one_t/T*100:.1f}% through the schedule)")
print(f"α_t at SNR=1  : {alpha[snr_one_idx]:.4f}")
print(f"σ_t at SNR=1  : {sigma[snr_one_idx]:.4f}")
print(f"ᾱ_t at SNR=1  : {alpha_bar[snr_one_idx]:.4f}  (should be ≈ 0.5)")

# Identify the three regimes
high_snr_t  = int(np.argmin(np.abs(snr - 10.0)))   # SNR ≈ 10: signal nearly intact
low_snr_t   = int(np.argmin(np.abs(snr - 0.1)))    # SNR ≈ 0.1: signal nearly gone

print(f"\nThree regimes:")
print(f"  High SNR (≈10, signal intact)  : t ≈ {high_snr_t}  "
      f"α={alpha[high_snr_t]:.3f}  σ={sigma[high_snr_t]:.3f}")
print(f"  Mid  SNR (≈1,  half-corrupted) : t ≈ {snr_one_idx}  "
      f"α={alpha[snr_one_idx]:.3f}  σ={sigma[snr_one_idx]:.3f}")
print(f"  Low  SNR (≈0.1, signal gone)   : t ≈ {low_snr_t}  "
      f"α={alpha[low_snr_t]:.3f}  σ={sigma[low_snr_t]:.3f}")

# ── Step 6: sanity checks ─────────────────────────────────────────────────
vp_check = alpha_bar + sigma**2   # should be 1 everywhere
print(f"\nVariance-preserving check  (ᾱ + σ² = 1):")
print(f"  max deviation from 1.0 : {np.abs(vp_check - 1.0).max():.2e}")
print(f"  ᾱ_0 = {alpha_bar[0]:.6f}  (should be 1.0)")
print(f"  ᾱ_T = {alpha_bar[T]:.6f}  (should be ≈ 0.0)")
print(f"  σ_0 = {sigma[0]:.6f}  (should be ≈ 0.0)")
print(f"  σ_T = {sigma[T]:.6f}  (should be ≈ 1.0)")
print(f"  β range: [{beta[1:].min():.2e}, {beta[1:].max():.4f}]")

# ── Step 7: plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: α_t and σ_t on linear scale ---
ax1 = axes[0]
ax1.plot(t_arr, alpha, color="tab:blue",   linewidth=2, label=r"$\alpha_t$ (signal)")
ax1.plot(t_arr, sigma, color="tab:red",    linewidth=2, label=r"$\sigma_t$ (noise)")
ax1.plot(t_arr, alpha_bar, color="tab:purple", linewidth=1.5,
         linestyle="--", label=r"$\bar{\alpha}_t$")
ax1.axvline(snr_one_t, color="gray", linewidth=1.2, linestyle=":",
            label=f"SNR = 1  (t = {snr_one_t:.0f})")
ax1.axvline(high_snr_t, color="tab:blue", linewidth=0.8, linestyle=":",
            alpha=0.5, label=f"SNR ≈ 10 (t = {high_snr_t})")
ax1.axvline(low_snr_t,  color="tab:red",  linewidth=0.8, linestyle=":",
            alpha=0.5, label=f"SNR ≈ 0.1 (t = {low_snr_t})")

# Shade the three regimes
ax1.axvspan(0,           high_snr_t,  alpha=0.06, color="tab:blue",
            label="High-SNR regime")
ax1.axvspan(high_snr_t,  low_snr_t,   alpha=0.06, color="tab:green",
            label="Mid-SNR regime")
ax1.axvspan(low_snr_t,   T,           alpha=0.06, color="tab:red",
            label="Low-SNR regime")

ax1.set_xlabel("Timestep t")
ax1.set_ylabel("Coefficient value")
ax1.set_title(f"Cosine noise schedule  (T={T}, s={s})\n"
              r"$\alpha_t^2 + \sigma_t^2 = 1$ at every t")
ax1.legend(fontsize=7.5, loc="center right")
ax1.set_xlim(0, T)
ax1.set_ylim(-0.02, 1.05)

# --- Right: SNR on log scale ---
ax2 = axes[1]
ax2.semilogy(t_arr, snr, color="darkorange", linewidth=2.5, label="SNR(t)")
ax2.axhline(1.0, color="gray", linewidth=1, linestyle="--",
            label="SNR = 1")
ax2.axhline(10.0, color="tab:blue", linewidth=0.8, linestyle=":",
            alpha=0.7, label="SNR = 10")
ax2.axhline(0.1, color="tab:red",  linewidth=0.8, linestyle=":",
            alpha=0.7, label="SNR = 0.1")
ax2.axvline(snr_one_t, color="gray", linewidth=1.2, linestyle=":",
            label=f"t = {snr_one_t:.0f}")
ax2.axvline(high_snr_t, color="tab:blue", linewidth=0.8, linestyle=":",
            alpha=0.5)
ax2.axvline(low_snr_t,  color="tab:red",  linewidth=0.8, linestyle=":",
            alpha=0.5)

ax2.axvspan(0,          high_snr_t, alpha=0.06, color="tab:blue")
ax2.axvspan(high_snr_t, low_snr_t,  alpha=0.06, color="tab:green")
ax2.axvspan(low_snr_t,  T,          alpha=0.06, color="tab:red")

ax2.set_xlabel("Timestep t")
ax2.set_ylabel("SNR(t)  [log scale]")
ax2.set_title("Signal-to-noise ratio\n"
              r"SNR(t) = $\bar{\alpha}_t$ / (1 − $\bar{\alpha}_t$)")
ax2.legend(fontsize=8)
ax2.set_xlim(0, T)
ax2.set_ylim(1e-3, 1e4)

plt.tight_layout()
plt.show()

# ── Step 8: forward process demo on a real residual ───────────────────────
# Pick one residual from the training set and show how it degrades
res_cols = [f"hist_res_{k:02d}" for k in range(15)]
rng_28   = np.random.default_rng(42)

sample_row = df_windows_train.sample(1, random_state=7).iloc[0]
r0 = sample_row[res_cols].values.astype(float)

demo_timesteps = [0, 25, snr_one_idx, 150, T]
LAT_CENTERS_28 = np.linspace(1.5, 43.5, 15)   # bin centers

fig2, axes2 = plt.subplots(1, len(demo_timesteps), figsize=(16, 4), sharey=True)
for ax, t_demo in zip(axes2, demo_timesteps):
    # Sample r_t = α_t * r_0 + σ_t * ε,  ε ~ N(0, I)
    eps      = rng_28.standard_normal(size=len(r0))
    r_t      = alpha[t_demo] * r0 + sigma[t_demo] * eps
    snr_demo = snr[t_demo]

    colors_d = ["tab:green" if v >= 0 else "tab:red" for v in r_t]
    ax.bar(LAT_CENTERS_28, r_t, width=2.8, color=colors_d, alpha=0.75)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(
        f"t = {t_demo}\n"
        f"SNR = {snr_demo:.2f}\n"
        f"α={alpha[t_demo]:.2f}  σ={sigma[t_demo]:.2f}",
        fontsize=8
    )
    ax.set_xlabel("Latitude (°)", fontsize=7)
    if t_demo == 0:
        ax.set_ylabel("Residual density")

fig2.suptitle(
    f"Forward process demo — residual corruption at selected timesteps\n"
    f"Cycle {int(sample_row['cycle'])} {sample_row['hemisphere']}  |  "
    f"τ = {sample_row['tau_center']:.2f} yr  |  "
    f"amplitude = {sample_row['amplitude']:.0f} MSH",
    fontsize=10
)
plt.tight_layout()
plt.show()

# ── Store schedule arrays for downstream tasks ────────────────────────────
schedule = dict(
    T         = T,
    s         = s,
    t_arr     = t_arr,
    alpha_bar = alpha_bar,
    alpha     = alpha,
    sigma     = sigma,
    beta      = beta,
    snr       = snr,
    snr_one_t = int(snr_one_t),
    high_snr_t= high_snr_t,
    low_snr_t = low_snr_t,
)
print(f"\n✓ Schedule stored in `schedule` dict — ready for Tasks 29+")
print(f"  Arrays: alpha_bar, alpha, sigma, beta, snr  (each length {T+1})")
print(f"  Key timesteps: SNR=1 at t={int(snr_one_t)}, "
      f"SNR=10 at t={high_snr_t}, SNR=0.1 at t={low_snr_t}")


---
## Task 29 — Forward corruption: from clean residual to noise level t in one step

The forward process is mathematically a **Markov chain** of T small noising
steps from t to t+1, but the closed-form expression r_t = α_t·r + σ_t·ε
lets you jump from t = 0 to *any* t in a single call. This is what makes
training tractable: at each gradient step you sample a random t, corrupt
the clean residual to that t directly, and ask the network to undo it. You
never simulate the chain.

**Tasks:**
- Implement a function `forward_corrupt(r, t, alpha, sigma, rng=None)`
  that takes a clean residual `r` of shape `(15,)`, an integer timestep
  `t`, the schedule arrays from Task 28, and an optional `numpy.random.Generator`
  for reproducibility, and returns:
  - `r_t` — the noisy residual, shape `(15,)`.
  - `eps` — the noise sample that was added, shape `(15,)`.
  Returning both is important for next week: the training objective will
  ask the network to predict `eps` from `r_t`, so we need both during
  training data generation.
- Add a vectorised variant `forward_corrupt_batch(R, t_array, alpha, sigma, rng=None)`
  that takes `R` of shape `(N, 15)` and `t_array` of shape `(N,)` (a
  *different* timestep for each row) and returns `(R_t, EPS)` of the same
  shape. This is the version next week's training loop will call.
- **Sanity check:** apply `forward_corrupt(r, 0, ...)` to a real residual.
  The output `r_t` should equal `r` to floating-point precision and `eps`
  can be anything (it will be multiplied by σ_0 ≈ 0). Apply
  `forward_corrupt(r, T, ...)`: the output should be dominated by `eps`
  and bear no obvious resemblance to `r`.


In [ ]:
# Put your code here for Task 29.
# Task 29: Forward corruption functions
# Depends on: schedule (from Task 28), df_windows_train (from Task 27)
# res_cols and LAT_CENTERS defined in Task 26/28

import numpy as np
import matplotlib.pyplot as plt

# ── Single-sample forward corruption ─────────────────────────────────────
def forward_corrupt(r, t, alpha, sigma, rng=None):
    """
    Corrupt a single clean residual to noise level t.

    Parameters
    ----------
    r     : np.ndarray, shape (15,)  — clean residual histogram
    t     : int                      — timestep in {0, …, T}
    alpha : np.ndarray, shape (T+1,) — signal coefficients from Task 28
    sigma : np.ndarray, shape (T+1,) — noise  coefficients from Task 28
    rng   : np.random.Generator | None — for reproducibility

    Returns
    -------
    r_t : np.ndarray, shape (15,)  — noisy residual at timestep t
    eps : np.ndarray, shape (15,)  — noise sample used (training target)
    """
    if rng is None:
        rng = np.random.default_rng()

    r   = np.asarray(r, dtype=np.float64)
    eps = rng.standard_normal(size=r.shape)
    r_t = alpha[t] * r + sigma[t] * eps
    return r_t, eps


# ── Vectorized batch forward corruption ──────────────────────────────────
def forward_corrupt_batch(R, t_array, alpha, sigma, rng=None):
    """
    Corrupt a batch of clean residuals, each to its own timestep.

    Parameters
    ----------
    R       : np.ndarray, shape (N, 15)  — batch of clean residuals
    t_array : np.ndarray, shape (N,)     — per-row timesteps (integers)
    alpha   : np.ndarray, shape (T+1,)
    sigma   : np.ndarray, shape (T+1,)
    rng     : np.random.Generator | None

    Returns
    -------
    R_t : np.ndarray, shape (N, 15)  — noisy residuals
    EPS : np.ndarray, shape (N, 15)  — noise samples (training targets)
    """
    if rng is None:
        rng = np.random.default_rng()

    R       = np.asarray(R, dtype=np.float64)
    t_array = np.asarray(t_array, dtype=int)

    # Broadcast: alpha_t and sigma_t need shape (N, 1) to multiply (N, 15)
    a_t = alpha[t_array][:, None]   # shape (N, 1)
    s_t = sigma[t_array][:, None]   # shape (N, 1)

    EPS = rng.standard_normal(size=R.shape)
    R_t = a_t * R + s_t * EPS
    return R_t, EPS


# ── Sanity check 1: t = 0 ────────────────────────────────────────────────
rng_29  = np.random.default_rng(42)
res_cols = [f"hist_res_{k:02d}" for k in range(15)]

sample_row = df_windows_train.sample(1, random_state=42).iloc[0]
r0 = sample_row[res_cols].values.astype(float)

r_t0, eps_t0 = forward_corrupt(r0, 0, alpha, sigma, rng=rng_29)
max_dev_t0   = np.abs(r_t0 - r0).max()

print("── Sanity check: t = 0 ──────────────────────────────────────────────")
print(f"  α_0 = {alpha[0]:.8f}  σ_0 = {sigma[0]:.2e}")
print(f"  max |r_t - r0| = {max_dev_t0:.2e}  (should be ≈ 0)")
print(f"  r_t0 ≈ r0: {np.allclose(r_t0, r0, atol=1e-6)}")

# ── Sanity check 2: t = T ────────────────────────────────────────────────
r_tT, eps_tT = forward_corrupt(r0, T, alpha, sigma, rng=rng_29)
signal_power = np.var(alpha[T] * r0)
noise_power  = np.var(sigma[T] * eps_tT)
corr_rT_r0   = float(np.corrcoef(r_tT, r0)[0, 1])

print("\n── Sanity check: t = T ──────────────────────────────────────────────")
print(f"  α_T = {alpha[T]:.2e}  σ_T = {sigma[T]:.6f}")
print(f"  Signal power (α_T·r0 variance): {signal_power:.2e}")
print(f"  Noise  power (σ_T·ε  variance): {noise_power:.4f}")
print(f"  Correlation(r_tT, r0): {corr_rT_r0:.4f}  (should be ≈ 0)")

# ── Sanity check 3: batch function ───────────────────────────────────────
print("\n── Sanity check: batch function ─────────────────────────────────────")
N_batch  = 64
R_batch  = df_windows_train.sample(N_batch, random_state=0)[res_cols].values
t_batch  = rng_29.integers(0, T + 1, size=N_batch)
R_t_batch, EPS_batch = forward_corrupt_batch(
    R_batch, t_batch, alpha, sigma, rng=rng_29)

print(f"  Input  shape : {R_batch.shape}")
print(f"  R_t    shape : {R_t_batch.shape}")
print(f"  EPS    shape : {EPS_batch.shape}")

# Verify: each row should satisfy r_t ≈ α[t]·r + σ[t]·ε
max_batch_err = max(
    np.abs(R_t_batch[i] - alpha[t_batch[i]] * R_batch[i]
           - sigma[t_batch[i]] * EPS_batch[i]).max()
    for i in range(N_batch)
)
print(f"  Max reconstruction error: {max_batch_err:.2e}  (should be 0)")

# Variance-preserving check across batch:
# Var[r_t] ≈ α_t² · Var[r] + σ_t² · 1  (if r and ε are independent)
print(f"  Batch t range: [{t_batch.min()}, {t_batch.max()}]")
print(f"  R_t row norms (mean ± std): "
      f"{np.linalg.norm(R_t_batch, axis=1).mean():.3f} ± "
      f"{np.linalg.norm(R_t_batch, axis=1).std():.3f}")

# ── Visualisation: corruption progression on one residual ─────────────────
demo_ts = [0, 20, schedule["snr_one_t"], 120, 160, T]
LAT_CENTERS_29 = np.linspace(1.5, 43.5, 15)

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=True)
axes_flat = axes.flatten()

rng_demo = np.random.default_rng(0)
for ax, t_demo in zip(axes_flat, demo_ts):
    r_demo, _ = forward_corrupt(r0, t_demo, alpha, sigma, rng=rng_demo)
    snr_demo  = snr[t_demo]

    colors_d = ["tab:green" if v >= 0 else "tab:red" for v in r_demo]
    ax.bar(LAT_CENTERS_29, r_demo, width=2.6, color=colors_d, alpha=0.75,
           label="Corrupted r_t")

    # Overlay clean residual as a line for reference
    ax.step(np.append(LAT_CENTERS_29 - 1.5, LAT_CENTERS_29[-1] + 1.5),
            np.append(r0, r0[-1]),
            color="black", linewidth=1.2, alpha=0.5,
            linestyle="--", label="Clean r_0" if t_demo == 0 else "_")

    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_title(
        f"t = {t_demo}  |  SNR = {snr_demo:.2f}\n"
        f"α = {alpha[t_demo]:.3f}   σ = {sigma[t_demo]:.3f}",
        fontsize=8.5
    )
    ax.set_xlabel("Latitude (°)", fontsize=8)

axes_flat[0].set_ylabel("Residual density")
axes_flat[3].set_ylabel("Residual density")
axes_flat[0].legend(fontsize=7)

fig.suptitle(
    f"Forward corruption — one residual through the noise schedule\n"
    f"Cycle {int(sample_row['cycle'])} {sample_row['hemisphere']}  "
    f"τ = {sample_row['tau_center']:.2f} yr  "
    f"amplitude = {sample_row['amplitude']:.0f} MSH",
    fontsize=10
)
plt.tight_layout()
plt.show()

# ── Visualisation: batch SNR distribution at training time ────────────────
# In each training step, t is sampled uniformly from {0, …, T}
# Show the distribution of SNR values a typical batch will see
fig2, axes3 = plt.subplots(1, 2, figsize=(13, 4))

n_train_steps = 1000
rng_snr = np.random.default_rng(1)
t_samples = rng_snr.integers(0, T + 1, size=n_train_steps)
snr_samples = snr[t_samples]

axes3[0].hist(t_samples, bins=40, color="tab:blue", alpha=0.7, density=True)
axes3[0].set_xlabel("Sampled timestep t")
axes3[0].set_ylabel("Density")
axes3[0].set_title(f"Timestep distribution over {n_train_steps} training steps\n"
                    "(uniform sampling — each t equally likely)")

axes3[1].hist(np.log10(snr_samples + 1e-6), bins=40,
              color="tab:orange", alpha=0.7, density=True)
axes3[1].set_xlabel("log₁₀ SNR(t)")
axes3[1].set_ylabel("Density")
axes3[1].axvline(0, color="black", linewidth=1, linestyle="--",
                 label="SNR = 1")
axes3[1].legend()
axes3[1].set_title("SNR distribution seen during training\n"
                    "(log scale — model trains across all SNR levels)")

plt.tight_layout()
plt.show()

print("\n✓ forward_corrupt and forward_corrupt_batch ready for Task 30")
print(f"  Single call  : forward_corrupt(r, t, alpha, sigma, rng)")
print(f"  Batch call   : forward_corrupt_batch(R, t_array, alpha, sigma, rng)")
print(f"  Both return  : (noisy_residual, noise_eps)")


---
## Task 30 — Visualize the forward trajectory

This is the headline visualization of the week. A picture of one residual
being progressively destroyed by the forward process is the conceptual
anchor for everything in Weeks 08 and 09. If a colleague who has never
seen diffusion asks you to explain what it is, this is the figure you
will show them.

**Tasks:**
- Pick one specific row from your training table that has a visually
  interesting residual — ideally one where the empirical histogram shows
  **bimodality** or asymmetric tails that the parametric Gaussian misses.
  Print its `(cycle, hemisphere, tau_center)` so the choice is reproducible.
- Apply `forward_corrupt` at six representative timesteps:
  `t ∈ {0, T/8, T/4, T/2, 3T/4, T}`. Use the *same* random seed for all
  six so the trajectory is deterministic and the comparison is fair —
  what changes between subplots is only the noise level, not the noise
  realization.
- Plot all six on a 2 × 3 grid of subplots. Each subplot shows:
  - the noisy residual r_t as a bar chart on the 15-bin latitude grid;
  - the SNR(t) value in the title.
  Use the same y-axis limits across all subplots so the eye can track the
  growing variance.
- Underneath, plot the **same six trajectories with a different random
  seed** and observe that at high SNR (low t) the trajectories agree
  closely, while at low SNR (high t) they diverge — different noise
  realizations produce visibly different r_t, even from the same r and t.
  This previews the intrinsic stochasticity of the forward process that
  Task 32 examines quantitatively.


In [ ]:
# Put your code here for Task 30.
# Task 30: Visualize the forward trajectory
# Depends on: forward_corrupt, schedule, df_windows_train, res_cols

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

LAT_CENTERS_30 = np.linspace(1.5, 43.5, 15)
BIN_WIDTH_30   = 3.0
res_cols       = [f"hist_res_{k:02d}" for k in range(15)]
emp_cols       = [f"hist_emp_{k:02d}" for k in range(15)]
par_cols       = [f"hist_par_{k:02d}" for k in range(15)]

# ── Step 1: pick an interesting row ──────────────────────────────────────
# Look for rows where the residual has large absolute values
# (bimodality / asymmetric tails — real distribution differs most from
# the parametric Gaussian)
df_windows_train["res_abs_sum"] = (
    df_windows_train[[f"hist_res_{k:02d}" for k in range(15)]]
    .abs().sum(axis=1)
)
# Among the top-10 most interesting residuals, pick one from mid-cycle
# (tau_center near 0) so the shape is physically interpretable
top10 = df_windows_train.nlargest(10, "res_abs_sum")
# Prefer mid-cycle (|tau| < 3) for visual interest
mid_cycle = top10[top10["tau_center"].abs() < 3]
chosen_row = mid_cycle.iloc[0] if len(mid_cycle) > 0 else top10.iloc[0]

r0 = chosen_row[res_cols].values.astype(float)
emp_vals = chosen_row[emp_cols].values.astype(float)
par_vals = chosen_row[par_cols].values.astype(float)

print("── Chosen row ───────────────────────────────────────────────────────")
print(f"  Cycle      : {int(chosen_row['cycle'])}  {chosen_row['hemisphere']}")
print(f"  tau_center : {chosen_row['tau_center']:.3f} yr")
print(f"  amplitude  : {chosen_row['amplitude']:.0f} MSH")
print(f"  n_obs      : {int(chosen_row['n_obs'])}")
print(f"  res_abs_sum: {chosen_row['res_abs_sum']:.4f}")
print(f"  window     : {chosen_row['window_start'].date()} – "
      f"{chosen_row['window_end'].date()}")

# ── Demo timesteps ────────────────────────────────────────────────────────
demo_ts = [
    0,
    T // 8,
    T // 4,
    T // 2,
    3 * T // 4,
    T,
]
labels_ts = [f"t={t}  SNR={snr[t]:.2f}" for t in demo_ts]

# ── Shared y-axis limits: accommodate r0 + max noise at t=T ──────────────
noise_scale = sigma[T]                     # ≈ 1 at t=T
r0_scale    = np.abs(r0).max()
ylim_abs    = max(r0_scale * 1.6, noise_scale * 3.5)
ylim        = (-ylim_abs, ylim_abs)

# ── Helper: one trajectory ────────────────────────────────────────────────
def make_trajectory(r0, demo_ts, seed):
    rng = np.random.default_rng(seed)
    results = []
    for t in demo_ts:
        r_t, eps = forward_corrupt(r0, t, alpha, sigma, rng=rng)
        results.append((t, r_t, eps))
    return results

traj_A = make_trajectory(r0, demo_ts, seed=0)
traj_B = make_trajectory(r0, demo_ts, seed=99)

# ── Color helpers ─────────────────────────────────────────────────────────
def bar_colors(vals, pos="tab:green", neg="tab:red"):
    return [pos if v >= 0 else neg for v in vals]

# ═══════════════════════════════════════════════════════════════════════════
# Figure 1: Seed A — the headline figure
# ═══════════════════════════════════════════════════════════════════════════
fig1 = plt.figure(figsize=(16, 8))
gs1  = gridspec.GridSpec(3, 3, figure=fig1,
                          height_ratios=[1.2, 1, 1],
                          hspace=0.55, wspace=0.35)

# --- Top row: clean empirical + parametric histograms (context) -----------
ax_emp = fig1.add_subplot(gs1[0, 0])
ax_par = fig1.add_subplot(gs1[0, 1])
ax_res = fig1.add_subplot(gs1[0, 2])

ax_emp.bar(LAT_CENTERS_30, emp_vals, width=BIN_WIDTH_30 * 0.8,
           color="steelblue", alpha=0.75)
ax_emp.set_title("Empirical histogram\n(what actually happened)", fontsize=8)
ax_emp.set_ylabel("Density")

ax_par.bar(LAT_CENTERS_30, par_vals, width=BIN_WIDTH_30 * 0.8,
           color="tomato", alpha=0.75)
ax_par.set_title("Parametric histogram\n(what the model predicted)", fontsize=8)

ax_res.bar(LAT_CENTERS_30, r0, width=BIN_WIDTH_30 * 0.8,
           color=bar_colors(r0), alpha=0.80)
ax_res.axhline(0, color="black", linewidth=0.8)
ax_res.set_title("Clean residual  r₀ = emp − par\n"
                  "(diffusion model's generation target)", fontsize=8)
for ax in [ax_emp, ax_par, ax_res]:
    ax.set_xlabel("Latitude (°)", fontsize=7)
    ax.set_xlim(0, 45)

# Mark the empirical and parametric peak latitudes
peak_emp = LAT_CENTERS_30[np.argmax(emp_vals)]
peak_par = LAT_CENTERS_30[np.argmax(par_vals)]
ax_emp.axvline(peak_emp, color="navy",   linewidth=1.2, linestyle="--",
               alpha=0.7, label=f"peak {peak_emp:.1f}°")
ax_par.axvline(peak_par, color="darkred", linewidth=1.2, linestyle="--",
               alpha=0.7, label=f"peak {peak_par:.1f}°")
ax_emp.legend(fontsize=7); ax_par.legend(fontsize=7)

# --- Middle row: corruption trajectory (seed A) ---------------------------
axes_mid = [fig1.add_subplot(gs1[1, col]) for col in range(3)]
axes_bot = [fig1.add_subplot(gs1[2, col]) for col in range(3)]

for row_axes, traj, seed_label in [
        (axes_mid, traj_A, "Seed A"),
        (axes_bot, traj_B, "Seed B")]:

    for ax, (t, r_t, eps) in zip(row_axes, traj[::2]):  # every other step: 0,T/4,T/2
        ax.bar(LAT_CENTERS_30, r_t, width=BIN_WIDTH_30 * 0.8,
               color=bar_colors(r_t), alpha=0.72)
        # Overlay clean r0 as a thin step line
        ax.step(np.append(LAT_CENTERS_30 - 1.5, LAT_CENTERS_30[-1] + 1.5),
                np.append(r0, r0[-1]),
                color="black", linewidth=1.0, alpha=0.35, linestyle="--")
        ax.axhline(0, color="black", linewidth=0.6)
        ax.set_ylim(ylim)
        ax.set_xlim(0, 45)
        ax.set_title(
            f"{seed_label}  |  t = {t}  ({t/T*100:.0f}%)\n"
            f"SNR = {snr[t]:.3f}   "
            f"α = {alpha[t]:.3f}   σ = {sigma[t]:.3f}",
            fontsize=7.5
        )
        ax.set_xlabel("Latitude (°)", fontsize=7)
        if ax == row_axes[0]:
            ax.set_ylabel("Residual density", fontsize=7)

fig1.suptitle(
    f"Forward corruption trajectory — Cycle {int(chosen_row['cycle'])} "
    f"{chosen_row['hemisphere']}  |  τ = {chosen_row['tau_center']:.2f} yr  |  "
    f"amplitude = {chosen_row['amplitude']:.0f} MSH\n"
    f"Top row: context histograms.  "
    f"Middle/bottom: same r₀ corrupted with two different random seeds.",
    fontsize=9, y=1.01
)
plt.savefig("task30_headline.png", dpi=150, bbox_inches="tight")
plt.show()

# ═══════════════════════════════════════════════════════════════════════════
# Figure 2: Full 2×3 grids — all six timesteps, both seeds
# ═══════════════════════════════════════════════════════════════════════════
for seed_label, traj in [("Seed A (seed=0)", traj_A),
                           ("Seed B (seed=99)", traj_B)]:

    fig2, axes2 = plt.subplots(2, 3, figsize=(15, 7),
                                sharey=True, sharex=True)
    axes2_flat  = axes2.flatten()

    for ax, (t, r_t, eps) in zip(axes2_flat, traj):
        ax.bar(LAT_CENTERS_30, r_t, width=BIN_WIDTH_30 * 0.78,
               color=bar_colors(r_t), alpha=0.75, label="r_t")
        ax.step(np.append(LAT_CENTERS_30 - 1.5, LAT_CENTERS_30[-1] + 1.5),
                np.append(r0, r0[-1]),
                color="black", linewidth=1.1, linestyle="--",
                alpha=0.40, label="r₀ (clean)")
        ax.axhline(0, color="black", linewidth=0.7)
        ax.set_ylim(ylim)
        ax.set_xlim(0, 45)
        ax.set_title(
            f"t = {t}  ({t/T*100:.0f}% through schedule)\n"
            f"SNR = {snr[t]:.3f}   α = {alpha[t]:.3f}   σ = {sigma[t]:.3f}",
            fontsize=8.5
        )
        ax.set_xlabel("Latitude (°)", fontsize=8)

    for ax in axes2[:, 0]:
        ax.set_ylabel("Residual density", fontsize=8)

    axes2_flat[0].legend(fontsize=7, loc="upper right")
    fig2.suptitle(
        f"Full forward trajectory — {seed_label}\n"
        f"Cycle {int(chosen_row['cycle'])} {chosen_row['hemisphere']}  |  "
        f"τ = {chosen_row['tau_center']:.2f} yr  |  "
        f"amplitude = {chosen_row['amplitude']:.0f} MSH  |  "
        f"n_obs = {int(chosen_row['n_obs'])}",
        fontsize=10
    )
    plt.tight_layout()
    plt.show()

# ═══════════════════════════════════════════════════════════════════════════
# Figure 3: Seed A vs Seed B — direct comparison at each timestep
# ═══════════════════════════════════════════════════════════════════════════
fig3, axes3 = plt.subplots(2, 3, figsize=(15, 6), sharey=True, sharex=True)

for col, ((t, r_tA, _), (_, r_tB, _)) in enumerate(zip(traj_A, traj_B)):
    diff     = r_tA - r_tB
    max_diff = np.abs(diff).max()
    snr_t    = snr[t]

    ax_top = axes3[0, col] if col < 3 else axes3[1, col - 3]
    ax_bot = axes3[1, col] if col < 3 else None

for col in range(3):
    t_A, r_tA, _ = traj_A[col]
    t_B, r_tB, _ = traj_B[col]
    diff          = r_tA - r_tB
    max_diff      = np.abs(diff).max()

    # Top row: overlay both trajectories
    ax_top = axes3[0, col]
    ax_top.bar(LAT_CENTERS_30 - 0.7, r_tA, width=BIN_WIDTH_30 * 0.45,
               color="tab:blue", alpha=0.65, label="Seed A")
    ax_top.bar(LAT_CENTERS_30 + 0.7, r_tB, width=BIN_WIDTH_30 * 0.45,
               color="tab:orange", alpha=0.65, label="Seed B")
    ax_top.axhline(0, color="black", linewidth=0.6)
    ax_top.set_ylim(ylim)
    ax_top.set_title(
        f"t = {t_A}  SNR = {snr[t_A]:.3f}\n"
        f"max|A−B| = {max_diff:.4f}",
        fontsize=8
    )
    if col == 0:
        ax_top.set_ylabel("Residual density")
        ax_top.legend(fontsize=7)

    # Bottom row: difference A − B
    ax_bot = axes3[1, col]
    ax_bot.bar(LAT_CENTERS_30, diff, width=BIN_WIDTH_30 * 0.75,
               color=bar_colors(diff, "tab:blue", "tab:orange"), alpha=0.75)
    ax_bot.axhline(0, color="black", linewidth=0.8)
    ax_bot.set_ylim(ylim)
    ax_bot.set_xlabel("Latitude (°)", fontsize=8)
    ax_bot.set_title(f"Difference A − B\nmax|diff| = {max_diff:.4f}", fontsize=8)
    if col == 0:
        ax_bot.set_ylabel("A − B")

fig3.suptitle(
    "Seed A vs Seed B — same r₀, same t, different noise realization\n"
    "High SNR (left): nearly identical.  "
    "Low SNR (right): completely different.  "
    "This stochasticity is what makes diffusion generative.",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ── Print the stochasticity numbers ──────────────────────────────────────
print("\n── Stochasticity across seeds ───────────────────────────────────────")
print(f"{'t':>5} {'SNR':>8} {'max|A-B|':>12} {'corr(A,B)':>12}")
print("-" * 42)
for (t, r_tA, _), (_, r_tB, _) in zip(traj_A, traj_B):
    diff     = r_tA - r_tB
    max_d    = np.abs(diff).max()
    corr_AB  = float(np.corrcoef(r_tA, r_tB)[0, 1])
    print(f"{t:>5} {snr[t]:>8.3f} {max_d:>12.4f} {corr_AB:>12.4f}")

print(f"\n✓ Task 30 complete — headline figure saved to task30_headline.png")
print(f"  Chosen row: cycle={int(chosen_row['cycle'])} "
      f"{chosen_row['hemisphere']}  tau={chosen_row['tau_center']:.3f}")


---
## Task 31 — Verify the limiting behaviour at t = T

A property of variance-preserving schedules is that as t → T, the
distribution of r_t converges to the standard Gaussian N(0, I) — and
crucially, this limit is **independent of the original data**. The
distribution at t = T does not "remember" what residuals you started from.
This is why next week's reverse process can begin from a fresh draw of
N(0, I) without any learning at the first step: at t = T, *all* clean
residuals corrupt to the same distribution, so we can sample that
distribution analytically rather than learning it.

This task asks you to verify the limiting behaviour empirically.

**Tasks:**
- Take the full **training set** of clean residuals (one residual per
  6-month window in the training cycles). Apply `forward_corrupt_batch`
  with `t_array = T` for every row — i.e., corrupt every training
  residual all the way to t = T, each with an independent fresh ε draw.
- For the resulting set of noised residuals at t = T:
  - Compute the **bin-wise mean** across all rows (a vector of length 15).
    Plot it as a bar chart. It should be statistically indistinguishable
    from zero, with bin-wise standard error roughly 1/√N where N is the
    number of training rows.
  - Compute the **bin-wise standard deviation** across all rows. Plot it
    as a bar chart. It should be statistically indistinguishable from 1,
    in every bin.
  - Compute the **bin-bin covariance matrix** (shape 15 × 15) and display
    it as a heatmap. It should look like the identity matrix to plotting
    precision: ~1 on the diagonal, ~0 off-diagonal.
- **Counter-test:** repeat the same three diagnostics on the *clean*
  training residuals (i.e., t = 0). The mean is non-zero in some bins,
  the standard deviation is bin-dependent, and the covariance matrix has
  off-diagonal structure. The contrast between the t = 0 and t = T
  diagnostics is exactly the structure the diffusion model has to learn
  to generate, run in reverse.


In [ ]:
# Put your code here for Task 31.
# Task 31: Verify the limiting behaviour at t = T
# Depends on: forward_corrupt_batch, schedule, df_windows_train, res_cols

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

res_cols      = [f"hist_res_{k:02d}" for k in range(15)]
LAT_CENTERS_31 = np.linspace(1.5, 43.5, 15)
BIN_WIDTH_31   = 3.0
N_BINS_31      = 15

# ── Step 1: extract clean training residuals ──────────────────────────────
R_clean = df_windows_train[res_cols].values.astype(np.float64)
N_train = len(R_clean)
print(f"Training residuals: {R_clean.shape}  (N={N_train} windows × 15 bins)")

# ── Step 2: corrupt all to t = T ─────────────────────────────────────────
rng_31   = np.random.default_rng(42)
t_all_T  = np.full(N_train, T, dtype=int)
R_noisy, EPS = forward_corrupt_batch(R_clean, t_all_T, alpha, sigma, rng=rng_31)

print(f"Corrupted residuals (t=T): {R_noisy.shape}")
print(f"α_T = {alpha[T]:.2e}   σ_T = {sigma[T]:.6f}")

# ── Step 3: bin-wise statistics at t = T ─────────────────────────────────
mean_noisy = R_noisy.mean(axis=0)          # shape (15,)
std_noisy  = R_noisy.std(axis=0)           # shape (15,)
cov_noisy  = np.cov(R_noisy.T)             # shape (15, 15)
se_noisy   = std_noisy / np.sqrt(N_train)  # standard error

# Expected: mean ≈ 0, std ≈ 1, cov ≈ I
print(f"\nt = T statistics:")
print(f"  Mean  — max abs: {np.abs(mean_noisy).max():.4f}  "
      f"(expected ≈ 0,  2σ threshold: {2/np.sqrt(N_train):.4f})")
print(f"  Std   — max |dev from 1|: {np.abs(std_noisy - 1).max():.4f}  "
      f"(expected ≈ 1)")
print(f"  Cov diagonal — mean: {np.diag(cov_noisy).mean():.4f}  "
      f"std: {np.diag(cov_noisy).std():.4f}  (expected ≈ 1)")
print(f"  Cov off-diag — mean abs: "
      f"{np.abs(cov_noisy - np.diag(np.diag(cov_noisy))).mean():.4f}  "
      f"(expected ≈ 0)")

# ── Step 4: bin-wise statistics at t = 0 (counter-test) ──────────────────
mean_clean = R_clean.mean(axis=0)
std_clean  = R_clean.std(axis=0)
cov_clean  = np.cov(R_clean.T)
se_clean   = std_clean / np.sqrt(N_train)

print(f"\nt = 0 statistics (counter-test):")
print(f"  Mean  — max abs: {np.abs(mean_clean).max():.4f}  "
      f"(should be non-zero in some bins)")
print(f"  Std   — max: {std_clean.max():.4f}  min: {std_clean.min():.4f}  "
      f"(should be bin-dependent, not ≈ 1)")
print(f"  Cov off-diag — mean abs: "
      f"{np.abs(cov_clean - np.diag(np.diag(cov_clean))).mean():.4f}  "
      f"(should show structure)")

# ── Step 5: normality test on the t=T residuals ───────────────────────────
from scipy.stats import shapiro, kstest, norm as sp_norm_31

print(f"\nNormality checks on t=T residuals:")
print(f"  {'Bin':>4}  {'KS p-val':>10}  {'Shapiro p-val':>14}  "
      f"{'mean':>8}  {'std':>8}")
for k in range(N_BINS_31):
    col_vals = R_noisy[:, k]
    ks_stat, ks_p   = kstest(col_vals, "norm", args=(0, 1))
    if N_train <= 5000:
        sh_stat, sh_p = shapiro(col_vals[:min(N_train, 5000)])
    else:
        sh_p = float("nan")
    print(f"  {k:>4}  {ks_p:>10.4f}  {sh_p:>14.4f}  "
          f"{col_vals.mean():>8.4f}  {col_vals.std():>8.4f}")

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: Mean and Std — t=0 vs t=T
# ══════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(2, 2, figsize=(14, 8))

# ── Top-left: mean at t=T ─────────────────────────────────────────────────
ax = axes1[0, 0]
ax.bar(LAT_CENTERS_31, mean_noisy, width=BIN_WIDTH_31 * 0.75,
       color=["tab:green" if v >= 0 else "tab:red" for v in mean_noisy],
       alpha=0.75)
ax.errorbar(LAT_CENTERS_31, mean_noisy, yerr=2 * se_noisy,
            fmt="none", color="black", capsize=3, linewidth=1.2,
            label="±2 SE")
ax.axhline(0, color="black", linewidth=1)
ax.set_title(f"Bin-wise mean at t = T\n"
             f"max|mean| = {np.abs(mean_noisy).max():.4f}  "
             f"(2σ threshold = {2/np.sqrt(N_train):.4f})",
             fontsize=9)
ax.set_ylabel("Mean residual density")
ax.set_xlabel("Latitude (°)")
ax.legend(fontsize=8)

# ── Top-right: mean at t=0 ────────────────────────────────────────────────
ax = axes1[0, 1]
ax.bar(LAT_CENTERS_31, mean_clean, width=BIN_WIDTH_31 * 0.75,
       color=["tab:green" if v >= 0 else "tab:red" for v in mean_clean],
       alpha=0.75)
ax.errorbar(LAT_CENTERS_31, mean_clean, yerr=2 * se_clean,
            fmt="none", color="black", capsize=3, linewidth=1.2,
            label="±2 SE")
ax.axhline(0, color="black", linewidth=1)
ax.set_title(f"Bin-wise mean at t = 0  (counter-test)\n"
             f"max|mean| = {np.abs(mean_clean).max():.4f}  "
             f"(non-zero = parametric model has consistent bias)",
             fontsize=9)
ax.set_ylabel("Mean residual density")
ax.set_xlabel("Latitude (°)")
ax.legend(fontsize=8)

# ── Bottom-left: std at t=T ────────────────────────────────────────────────
ax = axes1[1, 0]
ax.bar(LAT_CENTERS_31, std_noisy, width=BIN_WIDTH_31 * 0.75,
       color="steelblue", alpha=0.75)
ax.axhline(1.0, color="tab:red", linewidth=1.5, linestyle="--",
           label="Expected σ = 1")
ax.set_ylim(0, max(std_noisy.max(), 1.0) * 1.3)
ax.set_title(f"Bin-wise std at t = T\n"
             f"max|std − 1| = {np.abs(std_noisy - 1).max():.4f}  "
             f"(should be ≈ 1 everywhere)",
             fontsize=9)
ax.set_ylabel("Standard deviation")
ax.set_xlabel("Latitude (°)")
ax.legend(fontsize=8)

# ── Bottom-right: std at t=0 ──────────────────────────────────────────────
ax = axes1[1, 1]
ax.bar(LAT_CENTERS_31, std_clean, width=BIN_WIDTH_31 * 0.75,
       color="coral", alpha=0.75)
ax.axhline(1.0, color="black", linewidth=1.2, linestyle="--",
           label="σ = 1 reference")
ax.set_ylim(0, max(std_clean.max(), 1.0) * 1.3)
ax.set_title(f"Bin-wise std at t = 0  (counter-test)\n"
             f"range: [{std_clean.min():.4f}, {std_clean.max():.4f}]  "
             f"(bin-dependent, not ≈ 1)",
             fontsize=9)
ax.set_ylabel("Standard deviation")
ax.set_xlabel("Latitude (°)")
ax.legend(fontsize=8)

fig1.suptitle(
    "Limiting behaviour verification — t=T (left) vs t=0 (right)\n"
    "t=T should be N(0,I): zero mean, unit std everywhere.  "
    "t=0 shows the structure the diffusion model must learn.",
    fontsize=10, y=1.01
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: Covariance matrices — t=0 vs t=T
# ══════════════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, 3, figsize=(17, 5))

# Shared color scale: symmetric around 0
cov_absmax = max(np.abs(cov_clean).max(), np.abs(cov_noisy).max())
norm_cov   = TwoSlopeNorm(vmin=-cov_absmax, vcenter=0, vmax=cov_absmax)

# ── Covariance at t=0 ─────────────────────────────────────────────────────
im0 = axes2[0].imshow(cov_clean, cmap="RdBu_r", norm=norm_cov, aspect="auto")
axes2[0].set_title(
    f"Covariance at t = 0\n"
    f"Off-diag mean abs = "
    f"{np.abs(cov_clean - np.diag(np.diag(cov_clean))).mean():.4f}\n"
    f"(structured — what the model must learn)",
    fontsize=8.5
)
plt.colorbar(im0, ax=axes2[0], fraction=0.046, pad=0.04)
axes2[0].set_xlabel("Bin index"); axes2[0].set_ylabel("Bin index")

# ── Covariance at t=T ─────────────────────────────────────────────────────
im1 = axes2[1].imshow(cov_noisy, cmap="RdBu_r", norm=norm_cov, aspect="auto")
axes2[1].set_title(
    f"Covariance at t = T\n"
    f"Off-diag mean abs = "
    f"{np.abs(cov_noisy - np.diag(np.diag(cov_noisy))).mean():.4f}\n"
    f"(≈ identity — N(0,I) verified)",
    fontsize=8.5
)
plt.colorbar(im1, ax=axes2[1], fraction=0.046, pad=0.04)
axes2[1].set_xlabel("Bin index"); axes2[1].set_ylabel("Bin index")

# ── Difference: cov(t=0) − I ──────────────────────────────────────────────
diff_cov  = cov_clean - np.eye(N_BINS_31)
diff_amax = np.abs(diff_cov).max()
norm_diff = TwoSlopeNorm(vmin=-diff_amax, vcenter=0, vmax=diff_amax)
im2 = axes2[2].imshow(diff_cov, cmap="RdBu_r", norm=norm_diff, aspect="auto")
axes2[2].set_title(
    f"cov(t=0) − I\n"
    f"Max abs deviation: {diff_amax:.4f}\n"
    f"(structure the model must reconstruct from noise)",
    fontsize=8.5
)
plt.colorbar(im2, ax=axes2[2], fraction=0.046, pad=0.04)
axes2[2].set_xlabel("Bin index"); axes2[2].set_ylabel("Bin index")

for ax in axes2:
    ticks = np.arange(0, N_BINS_31, 3)
    tick_labels = [f"{LAT_CENTERS_31[i]:.0f}°" for i in ticks]
    ax.set_xticks(ticks); ax.set_xticklabels(tick_labels, fontsize=7)
    ax.set_yticks(ticks); ax.set_yticklabels(tick_labels, fontsize=7)

fig2.suptitle(
    "Covariance structure: t=0 (clean residuals) vs t=T (fully corrupted)\n"
    "t=T → identity matrix confirms N(0,I) convergence.  "
    "t=0 → correlated structure is the generation target.",
    fontsize=10
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 3: Distribution of individual bin values — Q-Q plots
# ══════════════════════════════════════════════════════════════════════════
from scipy.stats import probplot

fig3, axes3 = plt.subplots(3, 5, figsize=(16, 8))
axes3_flat  = axes3.flatten()

for k in range(N_BINS_31):
    ax = axes3_flat[k]
    # Q-Q plot of t=T values against N(0,1)
    (osm, osr), (slope, intercept, r_val) = probplot(
        R_noisy[:, k], dist="norm", plot=None)
    ax.scatter(osm, osr, s=3, alpha=0.4, color="steelblue", label="t=T")
    ax.plot(osm, slope * np.array(osm) + intercept,
            color="tab:red", linewidth=1.2, label="N(0,1)")
    ax.set_title(f"Bin {k}  ({LAT_CENTERS_31[k]:.0f}°)\n"
                 f"r²={r_val**2:.3f}", fontsize=7)
    ax.set_xlabel(""); ax.set_ylabel("")
    if k == 0:
        ax.legend(fontsize=6)

fig3.suptitle(
    "Q-Q plots: t=T corrupted residuals vs N(0,1)\n"
    "Points on the diagonal confirm each bin is individually Gaussian",
    fontsize=10
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 4: Intermediate timesteps — watch the convergence happen
# ══════════════════════════════════════════════════════════════════════════
check_ts    = [0, T//8, T//4, T//2, 3*T//4, T]
rng_conv    = np.random.default_rng(7)
mean_by_t   = []
std_by_t    = []
offdiag_by_t = []

for t_check in check_ts:
    t_arr_check = np.full(N_train, t_check, dtype=int)
    R_t, _      = forward_corrupt_batch(R_clean, t_arr_check, alpha, sigma,
                                         rng=rng_conv)
    mean_by_t.append(R_t.mean(axis=0))
    std_by_t.append(R_t.std(axis=0))
    cov_t = np.cov(R_t.T)
    offdiag_by_t.append(
        np.abs(cov_t - np.diag(np.diag(cov_t))).mean())

fig4, axes4 = plt.subplots(1, 3, figsize=(15, 4))

# Mean convergence: max|mean| vs t
max_mean_by_t = [np.abs(m).max() for m in mean_by_t]
axes4[0].plot(check_ts, max_mean_by_t, "o-", color="tab:purple", linewidth=2)
axes4[0].axhline(2/np.sqrt(N_train), color="gray", linestyle="--",
                  label=f"2/√N = {2/np.sqrt(N_train):.4f}")
axes4[0].set_xlabel("Timestep t")
axes4[0].set_ylabel("max |bin-wise mean|")
axes4[0].set_title("Mean convergence to 0\nas t → T")
axes4[0].legend(fontsize=8)

# Std convergence: max|std-1| vs t
max_std_dev_by_t = [np.abs(s - 1).max() for s in std_by_t]
axes4[1].plot(check_ts, max_std_dev_by_t, "o-", color="tab:orange", linewidth=2)
axes4[1].axhline(0, color="gray", linestyle="--", label="Target = 0")
axes4[1].set_xlabel("Timestep t")
axes4[1].set_ylabel("max |bin-wise std − 1|")
axes4[1].set_title("Std convergence to 1\nas t → T")
axes4[1].legend(fontsize=8)

# Off-diagonal covariance convergence
axes4[2].plot(check_ts, offdiag_by_t, "o-", color="tab:green", linewidth=2)
axes4[2].axhline(0, color="gray", linestyle="--", label="Target = 0")
axes4[2].set_xlabel("Timestep t")
axes4[2].set_ylabel("Mean abs off-diagonal covariance")
axes4[2].set_title("Covariance convergence to I\nas t → T")
axes4[2].legend(fontsize=8)

fig4.suptitle(
    "Convergence to N(0,I) as t increases from 0 to T\n"
    "All three statistics should reach their target values at t = T",
    fontsize=10
)
plt.tight_layout()
plt.show()

# ── Final summary ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  Task 31 Summary")
print("=" * 60)
print(f"  N training windows        : {N_train}")
print(f"  t=T mean   max|μ|         : {np.abs(mean_noisy).max():.5f}")
print(f"  t=T std    max|σ−1|       : {np.abs(std_noisy-1).max():.5f}")
print(f"  t=T cov    off-diag mean  : "
      f"{np.abs(cov_noisy - np.diag(np.diag(cov_noisy))).mean():.5f}")
print(f"  t=0 mean   max|μ|         : {np.abs(mean_clean).max():.5f}  ← non-zero")
print(f"  t=0 std    range          : "
      f"[{std_clean.min():.4f}, {std_clean.max():.4f}]  ← bin-dependent")
print(f"  t=0 cov    off-diag mean  : "
      f"{np.abs(cov_clean - np.diag(np.diag(cov_clean))).mean():.5f}  ← structured")
print("=" * 60)
print("\n✓ N(0,I) convergence verified — reverse process can safely")
print("  start from fresh N(0,I) samples at t=T.")
print("\n  The difference between t=0 and t=T statistics is the")
print("  full scope of what the diffusion model must learn.")

---
## Task 32 — The forward process is stochastic: many r_t's per (r, t)

A subtle point that often confuses students on first contact with
diffusion: the forward process is *not* a deterministic function of (r, t).
The same clean residual r corrupted at the same noise level t can produce
many different noisy residuals r_t — one per draw of ε. The forward
process maps each (r, t) to a **distribution** over r_t, not a single r_t.

This is why the reverse process must be a *learned* model rather than
a closed-form inverse: there is no inverse function, because the forward
process is many-to-one (in fact, many-to-many). The reverse model has to
estimate, for each noisy r_t, the most likely direction back toward the
clean data manifold — and "most likely" is a statement about a
distribution, which is what a neural network is good at approximating.

**Tasks:**
- Pick the same row you used in Task 30. Pick one intermediate timestep,
  say `t = T // 4`. Call `forward_corrupt(r, t, ...)` **100 times**, each
  with a different random seed. Collect the 100 noisy residuals into an
  array of shape `(100, 15)`.
- Plot all 100 noisy residuals on the same axes as faint lines, with the
  *clean* residual r overlaid as a thick black line. The cloud of faint
  lines should cluster around r with bin-wise spread proportional to σ_t.
- Compute the empirical mean and standard deviation across the 100
  samples in each bin, and overlay them as error bars on the clean
  residual. Verify that:
  - the empirical bin-wise mean ≈ α_t · r (the deterministic part of the
    corruption);
  - the empirical bin-wise standard deviation ≈ σ_t (the stochastic part).
- Repeat for t = T // 2 and t = 3T // 4. Watch the cloud spread out as
  σ_t grows and the deterministic centerline α_t·r shrinks toward zero.

This is the cleanest illustration of the forward process's structure
α_t (deterministic) + σ_t (stochastic) that you will see all week.


In [ ]:
# Put your code here for Task 32.
# Task 32: The forward process is stochastic — many r_t's per (r, t)
# Depends on: forward_corrupt, schedule, chosen_row (from Task 30), res_cols

import numpy as np
import matplotlib.pyplot as plt

res_cols       = [f"hist_res_{k:02d}" for k in range(15)]
LAT_CENTERS_32 = np.linspace(1.5, 43.5, 15)
BIN_WIDTH_32   = 3.0
N_SAMPLES      = 100   # noise draws per (r, t)

# ── Use the same row as Task 30 ───────────────────────────────────────────
r0 = chosen_row[res_cols].values.astype(float)
print(f"Using row: cycle={int(chosen_row['cycle'])} "
      f"{chosen_row['hemisphere']}  "
      f"tau={chosen_row['tau_center']:.3f}")

# ── Timesteps to examine ──────────────────────────────────────────────────
demo_ts_32 = [T // 4, T // 2, 3 * T // 4]
demo_labels = [f"t = T/4 = {T//4}", f"t = T/2 = {T//2}",
               f"t = 3T/4 = {3*T//4}"]

# ── Helper: generate N_SAMPLES corruptions of r0 at timestep t ───────────
def sample_forward_cloud(r0, t, n_samples, alpha, sigma, base_seed=0):
    """
    Returns array of shape (n_samples, 15) — independent corruptions
    of r0 at timestep t, each with a fresh random seed.
    """
    cloud = np.empty((n_samples, len(r0)))
    for i in range(n_samples):
        rng_i     = np.random.default_rng(base_seed + i)
        r_t, _    = forward_corrupt(r0, t, alpha, sigma, rng=rng_i)
        cloud[i]  = r_t
    return cloud

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: Cloud plots — all three timesteps in one figure
# ══════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(1, 3, figsize=(17, 5), sharey=False)

# Shared y limits: scale to max noise at t = 3T/4
sigma_max  = sigma[3 * T // 4]
r0_max     = np.abs(r0).max()
ylim_cloud = max(r0_max * 1.5, sigma_max * 4)

for ax, t_demo, label in zip(axes1, demo_ts_32, demo_labels):
    cloud = sample_forward_cloud(r0, t_demo, N_SAMPLES, alpha, sigma)

    # Expected deterministic center and stochastic spread
    center_expected = alpha[t_demo] * r0     # α_t · r
    std_expected    = sigma[t_demo]          # scalar

    # Empirical mean and std across the 100 samples
    mean_empirical  = cloud.mean(axis=0)     # shape (15,)
    std_empirical   = cloud.std(axis=0)      # shape (15,)

    # ── Plot 100 faint trajectories ───────────────────────────────────────
    for i in range(N_SAMPLES):
        ax.step(
            np.append(LAT_CENTERS_32 - 1.5, LAT_CENTERS_32[-1] + 1.5),
            np.append(cloud[i], cloud[i, -1]),
            color="steelblue", linewidth=0.4, alpha=0.12, where="post"
        )

    # ── Clean residual (thick black) ──────────────────────────────────────
    ax.step(
        np.append(LAT_CENTERS_32 - 1.5, LAT_CENTERS_32[-1] + 1.5),
        np.append(r0, r0[-1]),
        color="black", linewidth=2.0, alpha=0.9, where="post",
        label=r"Clean $r_0$"
    )

    # ── Deterministic center α_t · r (dashed green) ───────────────────────
    ax.step(
        np.append(LAT_CENTERS_32 - 1.5, LAT_CENTERS_32[-1] + 1.5),
        np.append(center_expected, center_expected[-1]),
        color="tab:green", linewidth=1.8, linestyle="--",
        alpha=0.9, where="post",
        label=rf"$\alpha_t \cdot r_0$  (deterministic)"
    )

    # ── Empirical mean ± 1 std (error bars, orange) ───────────────────────
    ax.errorbar(
        LAT_CENTERS_32, mean_empirical,
        yerr=std_empirical,
        fmt="o", color="tab:orange", markersize=4,
        capsize=3, linewidth=1.2, alpha=0.85,
        label=r"Empirical mean $\pm$ std"
    )

    ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
    ax.set_ylim(-ylim_cloud, ylim_cloud)
    ax.set_xlim(0, 45)
    ax.set_xlabel("Latitude (°)", fontsize=9)
    ax.set_title(
        f"{label}\n"
        f"α = {alpha[t_demo]:.3f}   σ = {sigma[t_demo]:.3f}   "
        f"SNR = {snr[t_demo]:.3f}",
        fontsize=9
    )
    if ax == axes1[0]:
        ax.set_ylabel("Residual density", fontsize=9)
        ax.legend(fontsize=7.5, loc="upper right")

fig1.suptitle(
    f"Forward process stochasticity — {N_SAMPLES} independent noise draws "
    f"at each timestep\n"
    f"Cycle {int(chosen_row['cycle'])} {chosen_row['hemisphere']}  |  "
    f"τ = {chosen_row['tau_center']:.2f} yr  |  "
    f"amplitude = {chosen_row['amplitude']:.0f} MSH\n"
    "Blue cloud = all 100 r_t samples.  "
    "Black = clean r₀.  Green dashed = α_t·r₀.  "
    "Orange = empirical mean ± std.",
    fontsize=9, y=1.02
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: Verification — empirical vs expected mean and std per bin
# ══════════════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(2, 3, figsize=(16, 7))

for col, (t_demo, label) in enumerate(zip(demo_ts_32, demo_labels)):
    cloud = sample_forward_cloud(r0, t_demo, N_SAMPLES, alpha, sigma)

    center_expected = alpha[t_demo] * r0
    std_expected    = np.full(15, sigma[t_demo])
    mean_empirical  = cloud.mean(axis=0)
    std_empirical   = cloud.std(axis=0)
    se              = std_empirical / np.sqrt(N_SAMPLES)

    # ── Top row: mean comparison ──────────────────────────────────────────
    ax_top = axes2[0, col]
    x      = np.arange(15)
    width  = 0.38
    ax_top.bar(x - width/2, mean_empirical, width=width,
               color="tab:orange", alpha=0.75, label="Empirical mean")
    ax_top.bar(x + width/2, center_expected, width=width,
               color="tab:green",  alpha=0.75, label=r"$\alpha_t \cdot r_0$ (expected)")
    ax_top.errorbar(x - width/2, mean_empirical, yerr=2*se,
                    fmt="none", color="black", capsize=2, linewidth=0.8)
    ax_top.axhline(0, color="black", linewidth=0.6)
    ax_top.set_title(
        f"{label} — Mean\n"
        f"max|emp − α·r| = "
        f"{np.abs(mean_empirical - center_expected).max():.4f}",
        fontsize=8.5
    )
    ax_top.set_xticks(x[::3])
    ax_top.set_xticklabels([f"{LAT_CENTERS_32[i]:.0f}°" for i in x[::3]],
                            fontsize=7)
    if col == 0:
        ax_top.set_ylabel("Residual density", fontsize=8)
        ax_top.legend(fontsize=7)

    # ── Bottom row: std comparison ────────────────────────────────────────
    ax_bot = axes2[1, col]
    ax_bot.bar(x - width/2, std_empirical, width=width,
               color="steelblue", alpha=0.75, label="Empirical std")
    ax_bot.bar(x + width/2, std_expected, width=width,
               color="tab:red",   alpha=0.55, label=r"$\sigma_t$ (expected)")
    ax_bot.set_title(
        f"{label} — Std\n"
        f"expected σ_t = {sigma[t_demo]:.4f}  |  "
        f"max|emp − σ_t| = {np.abs(std_empirical - sigma[t_demo]).max():.4f}",
        fontsize=8.5
    )
    ax_bot.set_xticks(x[::3])
    ax_bot.set_xticklabels([f"{LAT_CENTERS_32[i]:.0f}°" for i in x[::3]],
                            fontsize=7)
    if col == 0:
        ax_bot.set_ylabel("Standard deviation", fontsize=8)
        ax_bot.legend(fontsize=7)

fig2.suptitle(
    "Verification: empirical mean ≈ α_t·r₀  and  empirical std ≈ σ_t\n"
    "Top row: mean comparison per bin.  "
    "Bottom row: std comparison per bin.",
    fontsize=10
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 3: Signal vs noise decomposition as t grows
# ══════════════════════════════════════════════════════════════════════════
all_ts      = np.arange(0, T + 1)
signal_power = (alpha[all_ts] ** 2) * np.var(r0)
noise_power  = sigma[all_ts] ** 2    # variance of σ_t·ε per bin

fig3, axes3 = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: signal and noise power vs t ────────────────────────────────────
axes3[0].plot(all_ts, signal_power,
              color="tab:green", linewidth=2,
              label=r"Signal power: $\alpha_t^2 \cdot \mathrm{Var}(r_0)$")
axes3[0].plot(all_ts, noise_power,
              color="tab:red",   linewidth=2,
              label=r"Noise power: $\sigma_t^2$")
axes3[0].plot(all_ts, signal_power + noise_power,
              color="black",     linewidth=1.2, linestyle="--",
              alpha=0.6, label="Total (VP: should be constant)")

for t_demo in demo_ts_32:
    axes3[0].axvline(t_demo, color="gray", linewidth=0.8,
                     linestyle=":", alpha=0.7)
    axes3[0].text(t_demo + 1, signal_power.max() * 0.85,
                  f"T/{T//t_demo}", fontsize=7, color="gray")

axes3[0].set_xlabel("Timestep t")
axes3[0].set_ylabel("Power")
axes3[0].set_title("Signal vs noise power decomposition\n"
                    "Total is variance-preserving (dashed ≈ flat)")
axes3[0].legend(fontsize=8)

# ── Right: fraction of variance explained by signal vs noise ──────────────
total_power  = signal_power + noise_power
signal_frac  = signal_power / np.maximum(total_power, 1e-10)
noise_frac   = noise_power  / np.maximum(total_power, 1e-10)

axes3[1].stackplot(all_ts, signal_frac, noise_frac,
                   labels=["Signal fraction", "Noise fraction"],
                   colors=["tab:green", "tab:red"], alpha=0.65)
axes3[1].axhline(0.5, color="black", linewidth=1, linestyle="--",
                 alpha=0.6, label="50/50 split")
for t_demo in demo_ts_32:
    axes3[1].axvline(t_demo, color="white", linewidth=1.0,
                     linestyle=":", alpha=0.9)

axes3[1].set_xlabel("Timestep t")
axes3[1].set_ylabel("Fraction of total variance")
axes3[1].set_title("Signal vs noise fraction of total variance\n"
                    "Crossover = SNR = 1")
axes3[1].legend(fontsize=8, loc="center right")
axes3[1].set_xlim(0, T); axes3[1].set_ylim(0, 1)

fig3.suptitle(
    "Deterministic (signal) vs stochastic (noise) decomposition of r_t = α_t·r + σ_t·ε\n"
    "Vertical dotted lines mark the three demo timesteps.",
    fontsize=10
)
plt.tight_layout()
plt.show()

# ── Numeric verification table ────────────────────────────────────────────
print("\n── Verification table ───────────────────────────────────────────────")
print(f"{'t':>6} {'α_t':>7} {'σ_t':>7} "
      f"{'max|emp_mean − α·r|':>22} {'max|emp_std − σ_t|':>20} "
      f"{'corr(mean,α·r)':>16}")
print("-" * 82)
for t_demo in demo_ts_32:
    cloud      = sample_forward_cloud(r0, t_demo, N_SAMPLES, alpha, sigma,
                                       base_seed=1000)
    emp_mean   = cloud.mean(axis=0)
    emp_std    = cloud.std(axis=0)
    center     = alpha[t_demo] * r0
    max_mean_e = np.abs(emp_mean - center).max()
    max_std_e  = np.abs(emp_std - sigma[t_demo]).max()
    corr_mc    = float(np.corrcoef(emp_mean, center)[0, 1])
    print(f"{t_demo:>6} {alpha[t_demo]:>7.4f} {sigma[t_demo]:>7.4f} "
          f"{max_mean_e:>22.5f} {max_std_e:>20.5f} {corr_mc:>16.4f}")

print(f"\n  N samples per t: {N_SAMPLES}")
print(f"  Expected max|mean error| ≈ 2·σ_t/√N = "
      f"{2*sigma[T//2]/np.sqrt(N_SAMPLES):.4f} (at t=T/2)")
print(f"  Expected max|std  error| ≈ σ_t/√(2N) = "
      f"{sigma[T//2]/np.sqrt(2*N_SAMPLES):.4f} (at t=T/2)")

print("\n✓ Task 32 complete")
print("  Key insight: r_t = α_t·r₀ + σ_t·ε is a distribution, not a function.")
print("  The reverse model must learn p(r₀ | r_t) — a conditional distribution")
print("  — because many r_t values correspond to the same r₀ and vice versa.")


---
## Where Week 07 leaves us, and what Week 08 will need

By the end of Task 32 you have built every piece of diffusion-specific
machinery the model needs, in pure numpy:

- A **dataset** of residuals indexed by hemispheric cycle and 6-month
  window (Task 26).
- A **stratified train/val/test split** by cycle amplitude that respects
  the cycle-as-unit structure of the data (Task 27).
- A **cosine noise schedule** with all the bookkeeping (α, σ, SNR)
  needed by both training and sampling (Task 28).
- A **forward corruption function** that maps (r, t) → r_t in one step,
  vectorised over batches (Task 29).
- **Three independent verifications** that the forward process behaves
  the way the mathematics says it should (Tasks 30–32).

What you have **not** built is anything with learnable parameters. There
is no neural network in this notebook; there is no PyTorch in this
notebook; there is no training loop in this notebook. That is by design.
The forward process is where all the diffusion-specific math lives, and
you have implemented it transparently and verified it works.

Next week we will build the **reverse process**: a small denoising MLP,
trained in PyTorch + Lightning, whose job is to estimate ε from r_t. The
forward apparatus you built this week will be reused unchanged — it
generates the (r_t, t, ε) training triples the network learns from.
Nothing about Task 28's schedule, Task 29's forward function, or
Task 27's splits will change in Week 08. We are simply adding the
*learned* half of the model on top of the *numerical* half you have just
built.
